Cell 1 — Setup کامل

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import networkx as nx

PROJECT_ROOT = Path(".")

DATA_PROC = PROJECT_ROOT / "Data_proc"
DATA_ML = PROJECT_ROOT / "Data_ml"
GRAPH_DIR = DATA_ML / "graph_dataset"

DAY13_DIR = GRAPH_DIR / "day13_explainability"
DAY14_DIR = GRAPH_DIR / "day14_case_studies"
DAY15_DIR = GRAPH_DIR / "day15_external_validation"
DAY16_DIR = GRAPH_DIR / "day16_novel_predictions"
DAY17_DIR = GRAPH_DIR / "day17_novel_explainability"
DAY18_DIR = GRAPH_DIR / "day18_cancer_specific_analysis"

DAY19_DIR = GRAPH_DIR / "day19_paper_figures"
DAY19_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIRS = {
    "fig1": DAY19_DIR / "figure1_workflow",
    "fig2": DAY19_DIR / "figure2_dataset_construction",
    "fig3": DAY19_DIR / "figure3_embedding_representation",
    "fig4": DAY19_DIR / "figure4_model_progression",
    "fig5": DAY19_DIR / "figure5_graph_architecture",
    "fig6": DAY19_DIR / "figure6_explainability",
    "fig7": DAY19_DIR / "figure7_novel_discovery",
    "fig8": DAY19_DIR / "figure8_cancer_prioritization",
    "meta": DAY19_DIR / "figure_metadata",
}

for d in FIG_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 400
plt.rcParams["font.size"] = 11
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("DAY19_DIR:", DAY19_DIR)

Cell 2 — Load فایل‌های کلیدی

In [ ]:
files = {
    "pairs_all": DATA_PROC / "pairs" / "pairs_all_embedding_ready.csv",
    "final_pairs_qc": DATA_PROC / "qc_reports" / "final_pairs_qc.csv",
    "localization_qc": DATA_PROC / "qc_reports" / "localization_filter_qc.csv",
    "scrna_qc": DATA_PROC / "qc_reports" / "scrna_filter_qc.csv",
    "baseline": DATA_ML / "baseline_results" / "baseline_results_all.csv",
    "leaderboard": DATA_ML / "baseline_results" / "leaderboard_pr_auc.csv",
    "cross_model": DATA_ML / "cross_model_fusion_results" / "cross_model_fusion_results_all.csv",
    "nn_compare": DATA_ML / "neural_pair_scorer_results" / "nn_all_losses_final_comparison.csv",
    "graph_leaderboard": GRAPH_DIR / "day12_graph_final_leaderboard.csv",
    "graph_qc": GRAPH_DIR / "graph_full_dataset_qc.csv",
    "graph_nodes": GRAPH_DIR / "graph_nodes.csv",
    "interaction_edges": GRAPH_DIR / "interaction_edges_labeled.csv",
    "ppi_edges": GRAPH_DIR / "ppi_edges.csv",
    "loc_edges": GRAPH_DIR / "colocalization_edges.csv",
    "hgt_saved_summary": DAY13_DIR / "hgt_saved_summary.csv",
    "edge_importance": DAY13_DIR / "edge_type_importance_from_ablation.csv",
    "top_prediction_enrichment": DAY13_DIR / "top_prediction_enrichment.csv",
    "case_study": DAY14_DIR / "case_study_article_table.csv",
    "external_validation": DAY15_DIR / "uniprot_curated_validation_cases.csv",
    "novel_top500": DAY16_DIR / "top500_novel_predictions_hgt.csv",
    "novel_qc": DAY16_DIR / "day16_novel_prediction_qc.csv",
    "novel_explain_summary": DAY17_DIR / "novel_candidate_explainability_summary.csv",
    "relation_importance": DAY17_DIR / "novel_candidate_relation_importance_percent.csv",
    "day17_priority": DAY17_DIR / "day17_final_novel_candidate_priority_table.csv",
    "day18_priority": DAY18_DIR / "day18_final_gi_cancer_priority_candidates.csv",
    "driver_summary": DAY18_DIR / "driver_enrichment_summary.csv",
    "pathway_freq": DAY18_DIR / "gi_driver_pathway_frequency.csv",
    "crc_rank": DAY18_DIR / "crc_enzyme_rank.csv",
    "lihc_rank": DAY18_DIR / "lihc_enzyme_rank.csv",
    "gi_rank": DAY18_DIR / "gi_enzyme_rank.csv",
}

for k, p in files.items():
    print(k, "exists:", p.exists(), "|", p)

Cell 3 — Load امن فایل‌ها

In [ ]:
def safe_read_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    print("MISSING:", path)
    return None

data = {k: safe_read_csv(p) for k, p in files.items()}

for k, df in data.items():
    if df is not None:
        print(k, df.shape)

Cell 4 — ابزار مشترک ذخیره Figureها

In [ ]:
def save_figure(fig, outdir, name):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    fig.savefig(outdir / f"{name}.png", dpi=400, bbox_inches="tight")
    fig.savefig(outdir / f"{name}.pdf", bbox_inches="tight")
    print("saved:", outdir / f"{name}.png")
    print("saved:", outdir / f"{name}.pdf")

Cell 5 — Figure 1: Overall workflow

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis("off")

steps = [
    ("Raw data", "UbiBrowser\nBioGRID\nUniProt\nTISCH2"),
    ("Data curation", "Positive pairs\nNegative sampling\nPPI / localization / scRNA filters"),
    ("Protein representation", "ESM2 / ProtBERT\nPer-sequence\nPer-residue"),
    ("Pair-level learning", "XGBoost\nNeural scorer\nFusion"),
    ("Graph learning", "GraphSAGE\nHeteroGNN\nHGT"),
    ("Discovery", "Novel predictions\nExplainability\nCancer prioritization"),
]

x_positions = np.linspace(0.07, 0.93, len(steps))
y = 0.55

for i, (title, text) in enumerate(steps):
    x = x_positions[i]
    rect = patches.FancyBboxPatch(
        (x - 0.075, y - 0.16),
        0.15,
        0.32,
        boxstyle="round,pad=0.02",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black",
        transform=ax.transAxes,
    )
    ax.add_patch(rect)

    ax.text(x, y + 0.08, title, ha="center", va="center",
            fontsize=12, fontweight="bold", transform=ax.transAxes)
    ax.text(x, y - 0.04, text, ha="center", va="center",
            fontsize=10, transform=ax.transAxes)

    if i < len(steps) - 1:
        ax.annotate(
            "",
            xy=(x_positions[i+1] - 0.09, y),
            xytext=(x + 0.09, y),
            arrowprops=dict(arrowstyle="->", lw=1.8),
            xycoords=ax.transAxes,
        )

ax.text(
    0.5,
    0.9,
    "E3/DUB specificity prediction and discovery workflow",
    ha="center",
    va="center",
    fontsize=17,
    fontweight="bold",
    transform=ax.transAxes,
)

save_figure(fig, FIG_DIRS["fig1"], "figure1_overall_workflow")
plt.show()

Cell 6 — Figure 2: Dataset construction summary

In [ ]:
pairs_all = data["pairs_all"]
interaction_edges = data["interaction_edges"]
ppi_edges = data["ppi_edges"]
loc_edges = data["loc_edges"]
graph_nodes = data["graph_nodes"]

dataset_counts = pd.DataFrame([
    {"component": "Proteins", "count": len(graph_nodes)},
    {"component": "Positive pairs", "count": int((pairs_all["label"].astype(int) == 1).sum())},
    {"component": "Negative pairs", "count": int((pairs_all["label"].astype(int) == 0).sum())},
    {"component": "Interaction edges", "count": len(interaction_edges)},
    {"component": "PPI edges", "count": len(ppi_edges)},
    {"component": "Co-localization edges", "count": len(loc_edges)},
])

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(dataset_counts["component"], dataset_counts["count"])
ax.set_xlabel("Count")
ax.set_title("Dataset and graph construction summary", fontweight="bold")
ax.invert_yaxis()

for i, v in enumerate(dataset_counts["count"]):
    ax.text(v, i, f" {v:,}", va="center", fontsize=10)

plt.tight_layout()
save_figure(fig, FIG_DIRS["fig2"], "figure2_dataset_graph_summary")
plt.show()

dataset_counts.to_csv(FIG_DIRS["fig2"] / "figure2_dataset_counts.csv", index=False)

Cell 7 — Figure 3: Embedding and feature representation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.axis("off")

blocks = [
    ("Protein sequence", 0.12, 0.7),
    ("Protein language models\nProtBERT / ProtBERT-BFD / ESM2", 0.38, 0.7),
    ("Per-sequence embedding\nGlobal protein vector", 0.68, 0.82),
    ("Per-residue embedding\nResidue-level matrix", 0.68, 0.58),
    ("Pair features\nconcat / |diff| / product\nresidue similarity summaries", 0.42, 0.28),
    ("Pair classifier\nXGBoost / NN / Fusion", 0.74, 0.28),
]

for text, x, y in blocks:
    rect = patches.FancyBboxPatch(
        (x - 0.12, y - 0.08),
        0.24,
        0.16,
        boxstyle="round,pad=0.02",
        linewidth=1.4,
        facecolor="white",
        edgecolor="black",
        transform=ax.transAxes,
    )
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=10.5, transform=ax.transAxes)

arrows = [
    ((0.24, 0.7), (0.26, 0.7)),
    ((0.50, 0.72), (0.56, 0.82)),
    ((0.50, 0.68), (0.56, 0.58)),
    ((0.68, 0.74), (0.47, 0.36)),
    ((0.68, 0.50), (0.47, 0.36)),
    ((0.54, 0.28), (0.62, 0.28)),
]

for start, end in arrows:
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", lw=1.6),
                xycoords=ax.transAxes)

ax.text(
    0.5,
    0.93,
    "Protein representation and pair-level feature construction",
    ha="center",
    fontsize=16,
    fontweight="bold",
    transform=ax.transAxes,
)

save_figure(fig, FIG_DIRS["fig3"], "figure3_embedding_feature_representation")
plt.show()

Cell 8 — Figure 4: Model progression

In [ ]:
graph_lb = data["graph_leaderboard"].copy()

display(graph_lb.head())
print(graph_lb.columns.tolist())

Cell 9 — Figure 4: Model progression / performance comparison

In [ ]:
graph_lb = data["graph_leaderboard"].copy()

# چند مدل کلیدی برای شکل اصلی
model_order = [
    "GraphSAGE",
    "GAT",
    "HeteroGraphSAGE",
    "HeteroGAT",
    "EdgeAwareHetero",
    "HGT",
    "MegaWeighted",
]

# اگر اسم دقیق مدل‌ها متفاوت بود، با جستجوی تقریبی match می‌کنیم
def find_model_row(df, keyword):
    hit = df[df["model"].astype(str).str.lower().str.contains(keyword.lower(), regex=False)]
    if len(hit) == 0:
        return None
    return hit.sort_values("pr_auc_mean", ascending=False).iloc[0]

rows = []

keywords = {
    "GraphSAGE": "graphsage",
    "GAT": "gat",
    "HeteroGraphSAGE": "hetero_graphsage",
    "HeteroGAT": "hetero_gat",
    "EdgeAwareHetero": "edge",
    "HGT": "hgt",
    "MegaWeighted": "mega",
}

for label, key in keywords.items():
    r = find_model_row(graph_lb, key)
    if r is not None:
        rows.append({
            "model_label": label,
            "model": r["model"],
            "roc_auc_mean": r["roc_auc_mean"],
            "roc_auc_std": r["roc_auc_std"],
            "pr_auc_mean": r["pr_auc_mean"],
            "pr_auc_std": r["pr_auc_std"],
        })

plot_df = pd.DataFrame(rows)

display(plot_df)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(plot_df))

ax.errorbar(
    x,
    plot_df["pr_auc_mean"],
    yerr=plot_df["pr_auc_std"],
    fmt="o-",
    linewidth=2.2,
    capsize=4,
    label="PR-AUC"
)

ax.errorbar(
    x,
    plot_df["roc_auc_mean"],
    yerr=plot_df["roc_auc_std"],
    fmt="s--",
    linewidth=2.0,
    capsize=4,
    label="ROC-AUC"
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["model_label"], rotation=30, ha="right")
ax.set_ylabel("Cross-validation performance")
ax.set_title("Progression of graph learning models", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

save_figure(fig, FIG_DIRS["fig4"], "figure4_graph_model_progression")
plt.show()

plot_df.to_csv(FIG_DIRS["fig4"] / "figure4_model_progression_data.csv", index=False)

Cell 10 — Figure 5: Graph architecture and edge composition

In [ ]:
graph_nodes = data["graph_nodes"]
interaction_edges = data["interaction_edges"]
ppi_edges = data["ppi_edges"]
loc_edges = data["loc_edges"]

edge_comp = pd.DataFrame([
    {"edge_type": "Enzyme-substrate", "count": len(interaction_edges)},
    {"edge_type": "PPI", "count": len(ppi_edges)},
    {"edge_type": "Co-localization", "count": len(loc_edges)},
])

fig, ax = plt.subplots(figsize=(7, 6))

ax.pie(
    edge_comp["count"],
    labels=edge_comp["edge_type"],
    autopct=lambda p: f"{p:.1f}%",
    startangle=90
)

ax.set_title(
    f"Heterogeneous biological graph\n{len(graph_nodes):,} proteins, {edge_comp['count'].sum():,} edges",
    fontweight="bold"
)

plt.tight_layout()
save_figure(fig, FIG_DIRS["fig5"], "figure5_graph_edge_composition")
plt.show()

edge_comp.to_csv(FIG_DIRS["fig5"] / "figure5_edge_composition_data.csv", index=False)

Cell 11 — Figure 6: Explainability / relation importance

In [ ]:
relation_imp = data["relation_importance"].copy()

display(relation_imp.head())
print(relation_imp.columns.tolist())

Cell 12 — Figure 6: Explainability (Relation Importance)

In [ ]:
relation_imp = data["relation_importance"].copy()

mean_importance = pd.DataFrame({
    "Relation":[
        "PPI",
        "Co-localization",
        "Interaction context"
    ],
    "Importance (%)":[
        relation_imp["ppi_importance_pct"].mean(),
        relation_imp["colocalization_importance_pct"].mean(),
        relation_imp["enzyme_substrate_context_importance_pct"].mean()
    ]
})

display(mean_importance)

fig, ax = plt.subplots(figsize=(7,6))

bars = ax.bar(
    mean_importance["Relation"],
    mean_importance["Importance (%)"]
)

ax.set_ylim(0,100)
ax.set_ylabel("Average contribution (%)")
ax.set_title("Contribution of biological relations to HGT predictions",
             fontweight="bold")

for b in bars:
    h = b.get_height()
    ax.text(
        b.get_x()+b.get_width()/2,
        h+1,
        f"{h:.1f}",
        ha="center",
        fontsize=11
    )

plt.tight_layout()

save_figure(
    fig,
    FIG_DIRS["fig6"],
    "figure6_relation_importance"
)

plt.show()

mean_importance.to_csv(
    FIG_DIRS["fig6"]/"figure6_relation_importance.csv",
    index=False
)

Cell 13 — Figure 7: Novel prediction pipeline

In [ ]:
novel = data["day17_priority"].copy()

display(novel.head())

fig, ax = plt.subplots(figsize=(10,6))

ax.axis("off")

pipeline = [
    "All candidate\npairs",
    "HGT\nprediction",
    "Top novel\npredictions",
    "Explainability",
    "Literature\nsupport",
    "Final\npriority"
]

xs = np.linspace(0.08,0.92,len(pipeline))

for x,label in zip(xs,pipeline):

    rect = patches.FancyBboxPatch(
        (x-0.07,0.42),
        0.14,
        0.18,
        boxstyle="round,pad=0.02",
        linewidth=1.5,
        facecolor="white"
    )

    ax.add_patch(rect)

    ax.text(
        x,
        0.51,
        label,
        ha="center",
        va="center",
        fontsize=11
    )

for i in range(len(xs)-1):

    ax.annotate(
        "",
        xy=(xs[i+1]-0.08,0.51),
        xytext=(xs[i]+0.08,0.51),
        arrowprops=dict(
            arrowstyle="->",
            lw=2
        )
    )

ax.set_title(
    "Novel interaction discovery workflow",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

save_figure(
    fig,
    FIG_DIRS["fig7"],
    "figure7_novel_prediction_pipeline"
)

plt.show()

Cell 14 — Figure 8: Cancer prioritization pathways

In [ ]:
pathway = data["pathway_freq"].copy()

display(pathway)

fig, ax = plt.subplots(figsize=(9,6))

pathway = pathway.sort_values(
    "count",
    ascending=True
)

bars = ax.barh(
    pathway["pathway"],
    pathway["count"]
)

ax.set_xlabel("Number of prioritized interactions")
ax.set_title(
    "Cancer driver pathway enrichment",
    fontweight="bold"
)

for b in bars:

    ax.text(
        b.get_width()+1,
        b.get_y()+b.get_height()/2,
        str(int(b.get_width())),
        va="center"
    )

plt.tight_layout()

save_figure(
    fig,
    FIG_DIRS["fig8"],
    "figure8_driver_pathways"
)

plt.show()

pathway.to_csv(
    FIG_DIRS["fig8"]/"figure8_driver_pathways.csv",
    index=False
)

Cell جدید 1 — نصب/لود پکیج‌های حرفه‌ای

In [ ]:
!pip install seaborn

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import networkx as nx

from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D

PROJECT_ROOT = Path(".")

DATA_ML = PROJECT_ROOT / "Data_ml"
DATA_PROC = PROJECT_ROOT / "Data_proc"
GRAPH_DIR = DATA_ML / "graph_dataset"

DAY13_DIR = GRAPH_DIR / "day13_explainability"
DAY16_DIR = GRAPH_DIR / "day16_novel_predictions"
DAY17_DIR = GRAPH_DIR / "day17_novel_explainability"
DAY18_DIR = GRAPH_DIR / "day18_cancer_specific_analysis"

DAY19_DIR = GRAPH_DIR / "day19_paper_figures_v2"
DAY19_DIR.mkdir(parents=True, exist_ok=True)

for i in range(1, 9):
    (DAY19_DIR / f"figure{i}").mkdir(parents=True, exist_ok=True)

sns.set_theme(
    context="paper",
    style="whitegrid",
    font_scale=1.15
)

mpl.rcParams["figure.dpi"] = 140
mpl.rcParams["savefig.dpi"] = 500
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["axes.linewidth"] = 1.2
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["font.family"] = "DejaVu Sans"

print("DAY19 output:", DAY19_DIR)

Cell جدید 2 — تابع ذخیره حرفه‌ای

In [ ]:
def save_pubfig(fig, fig_dir, filename):
    fig_dir = Path(fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)

    png_path = fig_dir / f"{filename}.png"
    pdf_path = fig_dir / f"{filename}.pdf"
    svg_path = fig_dir / f"{filename}.svg"

    fig.savefig(png_path, dpi=500, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")

    print("saved:")
    print(png_path)
    print(pdf_path)
    print(svg_path)

Cell جدید 3 — لود همه جدول‌های مدل‌ها

In [ ]:
baseline = pd.read_csv(DATA_ML / "baseline_results" / "baseline_results_all.csv")
leaderboard = pd.read_csv(DATA_ML / "baseline_results" / "leaderboard_pr_auc.csv")

cross_model = pd.read_csv(
    DATA_ML / "cross_model_fusion_results" / "cross_model_fusion_results_all.csv"
)

nn_compare = pd.read_csv(
    DATA_ML / "neural_pair_scorer_results" / "nn_all_losses_final_comparison.csv"
)

graph_leaderboard = pd.read_csv(
    GRAPH_DIR / "day12_graph_final_leaderboard.csv"
)

mega = pd.read_csv(
    GRAPH_DIR / "mega_weighted_ensemble_results.csv"
)

print("baseline:", baseline.shape)
print("cross_model:", cross_model.shape)
print("nn_compare:", nn_compare.shape)
print("graph_leaderboard:", graph_leaderboard.shape)
print("mega:", mega.shape)

display(baseline.head())
display(cross_model.head())
display(nn_compare.head())
display(graph_leaderboard.head())
display(mega.head())

Cell جدید 4 — بررسی ستون‌ها

In [ ]:
for name, df in {
    "baseline": baseline,
    "cross_model": cross_model,
    "nn_compare": nn_compare,
    "graph_leaderboard": graph_leaderboard,
    "mega": mega,
}.items():
    print("\n" + "="*80)
    print(name)
    print(df.columns.tolist())

Cell 5 — آماده‌سازی جدول یکپارچه مدل‌ها

In [ ]:
model_rows = []

# Baseline / XGBoost / classical
tmp = baseline.copy()
tmp["family"] = "Pair-level ML"
tmp["model_name"] = (
    tmp["embedding_model"].astype(str)
    + " | "
    + tmp["feature_type"].astype(str)
    + " | "
    + tmp["algorithm"].astype(str)
)
model_rows.append(
    tmp[
        [
            "family", "model_name",
            "roc_auc_mean", "roc_auc_std",
            "pr_auc_mean", "pr_auc_std",
            "f1_mean", "accuracy_mean",
            "precision_mean", "recall_mean", "brier_mean"
        ]
    ]
)

# Cross-model fusion
tmp = cross_model.copy()
tmp["family"] = "Cross-model fusion"
tmp["model_name"] = tmp["fusion_name"].astype(str) + " | " + tmp["algorithm"].astype(str)
model_rows.append(
    tmp[
        [
            "family", "model_name",
            "roc_auc_mean", "roc_auc_std",
            "pr_auc_mean", "pr_auc_std",
            "f1_mean", "accuracy_mean",
            "precision_mean", "recall_mean", "brier_mean"
        ]
    ]
)

# Neural models
tmp = nn_compare.copy()
tmp["family"] = "Neural pair scorer"
tmp["model_name"] = tmp["model"].astype(str)
tmp["roc_auc_std"] = np.nan
tmp["pr_auc_std"] = np.nan
model_rows.append(
    tmp[
        [
            "family", "model_name",
            "roc_auc_mean", "roc_auc_std",
            "pr_auc_mean", "pr_auc_std",
            "f1_mean", "accuracy_mean",
            "precision_mean", "recall_mean", "brier_mean"
        ]
    ]
)

# Graph models
tmp = graph_leaderboard.copy()
tmp["family"] = "Graph learning"
tmp["model_name"] = tmp["model"].astype(str)
model_rows.append(
    tmp[
        [
            "family", "model_name",
            "roc_auc_mean", "roc_auc_std",
            "pr_auc_mean", "pr_auc_std",
            "f1_mean", "accuracy_mean",
            "precision_mean", "recall_mean", "brier_mean"
        ]
    ]
)

# Mega ensemble
tmp = mega.copy()
tmp["family"] = "Final ensemble"
tmp["model_name"] = tmp["model"].astype(str)
model_rows.append(
    tmp[
        [
            "family", "model_name",
            "roc_auc_mean", "roc_auc_std",
            "pr_auc_mean", "pr_auc_std",
            "f1_mean", "accuracy_mean",
            "precision_mean", "recall_mean", "brier_mean"
        ]
    ]
)

all_models = pd.concat(model_rows, ignore_index=True)

all_models = all_models.sort_values(
    "pr_auc_mean",
    ascending=False
).reset_index(drop=True)

all_models["rank"] = np.arange(1, len(all_models) + 1)

display(all_models.head(25))

all_models.to_csv(
    DAY19_DIR / "figure4" / "all_model_comparison_unified.csv",
    index=False
)

Cell 6 — Figure 4A/B/C حرفه‌ای: مدل‌ها، خانواده‌ها، متریک‌ها

In [ ]:
top_n = 20
plot_top = all_models.head(top_n).copy()
plot_top = plot_top.sort_values("pr_auc_mean", ascending=True)

palette = {
    "Pair-level ML": "#8da0cb",
    "Cross-model fusion": "#66c2a5",
    "Neural pair scorer": "#fc8d62",
    "Graph learning": "#e78ac3",
    "Final ensemble": "#ffd92f",
}

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(
    2, 2,
    figure=fig,
    height_ratios=[1.4, 1],
    width_ratios=[1.25, 1],
    hspace=0.35,
    wspace=0.28
)

# Panel A: Top models by PR-AUC
ax1 = fig.add_subplot(gs[:, 0])

sns.barplot(
    data=plot_top,
    y="model_name",
    x="pr_auc_mean",
    hue="family",
    dodge=False,
    palette=palette,
    ax=ax1
)

ax1.errorbar(
    plot_top["pr_auc_mean"],
    np.arange(len(plot_top)),
    xerr=plot_top["pr_auc_std"].fillna(0),
    fmt="none",
    ecolor="black",
    elinewidth=1,
    capsize=3
)

ax1.set_xlabel("PR-AUC")
ax1.set_ylabel("")
ax1.set_title("A. Top-performing models ranked by PR-AUC", loc="left", fontweight="bold")
ax1.legend(title="Model family", loc="lower right", frameon=True)
ax1.set_xlim(max(0, plot_top["pr_auc_mean"].min() - 0.03), min(1.0, plot_top["pr_auc_mean"].max() + 0.02))

# Panel B: Best model per family
ax2 = fig.add_subplot(gs[0, 1])

best_family = (
    all_models
    .sort_values("pr_auc_mean", ascending=False)
    .groupby("family", as_index=False)
    .first()
)

order = best_family.sort_values("pr_auc_mean", ascending=False)["family"]

sns.barplot(
    data=best_family,
    x="pr_auc_mean",
    y="family",
    order=order,
    palette=palette,
    ax=ax2
)

ax2.set_xlabel("Best PR-AUC")
ax2.set_ylabel("")
ax2.set_title("B. Best model within each family", loc="left", fontweight="bold")

for i, (_, r) in enumerate(best_family.set_index("family").loc[order].reset_index().iterrows()):
    ax2.text(
        r["pr_auc_mean"] + 0.004,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=10
    )

ax2.set_xlim(0, 1)

# Panel C: Metric profile of final model
ax3 = fig.add_subplot(gs[1, 1])

final_row = all_models.iloc[0].copy()

metric_df = pd.DataFrame({
    "metric": ["ROC-AUC", "PR-AUC", "F1", "Accuracy", "Precision", "Recall"],
    "value": [
        final_row["roc_auc_mean"],
        final_row["pr_auc_mean"],
        final_row["f1_mean"],
        final_row["accuracy_mean"],
        final_row["precision_mean"],
        final_row["recall_mean"],
    ]
})

sns.barplot(
    data=metric_df,
    x="metric",
    y="value",
    color="#4c72b0",
    ax=ax3
)

ax3.set_ylim(0, 1)
ax3.set_xlabel("")
ax3.set_ylabel("Score")
ax3.set_title("C. Metric profile of the best model", loc="left", fontweight="bold")
ax3.tick_params(axis="x", rotation=30)

for i, r in metric_df.iterrows():
    ax3.text(
        i,
        r["value"] + 0.015,
        f"{r['value']:.3f}",
        ha="center",
        fontsize=10
    )

fig.suptitle(
    "Comprehensive model benchmarking across pair-level, neural, graph, and ensemble models",
    fontsize=17,
    fontweight="bold",
    y=0.98
)

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4_comprehensive_model_benchmark"
)

plt.show()

Cell 7 — Figure 4D: Heatmap تمام متریک‌های Top15

In [ ]:
top_metric = all_models.head(15).copy()

heat_cols = [
    "roc_auc_mean",
    "pr_auc_mean",
    "f1_mean",
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
]

heat_df = top_metric.set_index("model_name")[heat_cols].copy()
heat_df.columns = ["ROC-AUC", "PR-AUC", "F1", "Accuracy", "Precision", "Recall"]

fig, ax = plt.subplots(figsize=(12, 8))

sns.heatmap(
    heat_df,
    annot=True,
    fmt=".3f",
    cmap="viridis",
    linewidths=0.5,
    cbar_kws={"label": "Performance"},
    ax=ax
)

ax.set_title(
    "D. Multi-metric performance heatmap of top models",
    fontweight="bold",
    loc="left"
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4_top_models_metric_heatmap"
)

plt.show()

Cell 5 — جدول تمیز و label کوتاه برای مدل‌ها

In [ ]:
import textwrap

def short_model_label(row):
    family = row["family"]
    name = str(row["model_name"])

    if family == "Final ensemble":
        return "MegaWeighted\nHGT+EdgeAware+HeteroSAGE"

    if "hgt" in name.lower() and family == "Graph learning":
        return "HGT"

    if "edge" in name.lower():
        return "Edge-aware\nHeteroGNN"

    if "hetero_graphsage" in name.lower() or "heterographsage" in name.lower():
        return "HeteroGraphSAGE"

    if "hetero_gat" in name.lower() or "heterogat" in name.lower():
        return "HeteroGAT"

    if "graphsage" in name.lower():
        return "GraphSAGE"

    if "gat" in name.lower():
        return "GAT"

    if family == "Neural pair scorer":
        return name.replace("_", "\n")

    if family == "Cross-model fusion":
        if "esm650" in name.lower() and "protbert" in name.lower():
            return "Cross-model\nESM650+ProtBERT"
        if "esm650" in name.lower() and "esm3b" in name.lower():
            return "Cross-model\nESM650+ESM3B"
        return "Cross-model\nFusion"

    if family == "Pair-level ML":
        if "xgb" in name.lower() or "xgboost" in name.lower():
            return "XGBoost\nPair Fusion"
        if "logistic" in name.lower():
            return "Logistic\nPair Fusion"
        return "Pair-level\nML"

    return "\n".join(textwrap.wrap(name, width=18))


all_models_clean = all_models.copy()
all_models_clean["short_label"] = all_models_clean.apply(short_model_label, axis=1)

# حذف labelهای تکراری و نگه داشتن بهترین performance از هر label
all_models_clean = (
    all_models_clean
    .sort_values("pr_auc_mean", ascending=False)
    .groupby(["family", "short_label"], as_index=False)
    .first()
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

display(all_models_clean.head(25))

all_models_clean.to_csv(
    DAY19_DIR / "figure4" / "all_model_comparison_clean_labels.csv",
    index=False
)

Cell 6 — Figure 4A حرفه‌ای و خوانا: فقط مدل‌های کلیدی

In [ ]:
key_labels = [
    "MegaWeighted\nHGT+EdgeAware+HeteroSAGE",
    "HGT",
    "Edge-aware\nHeteroGNN",
    "HeteroGraphSAGE",
    "GraphSAGE",
    "Cross-model\nESM650+ProtBERT",
    "Cross-model\nESM650+ESM3B",
    "NN\nFocal",
    "XGBoost\nPair Fusion",
    "Logistic\nPair Fusion",
]

plot_df = all_models_clean[
    all_models_clean["short_label"].isin(key_labels)
].copy()

# اگر بعضی labelها دقیقاً match نشدند، topها را اضافه می‌کنیم
if len(plot_df) < 7:
    plot_df = all_models_clean.head(12).copy()

plot_df = plot_df.sort_values("pr_auc_mean", ascending=True)

family_palette = {
    "Pair-level ML": "#4E79A7",
    "Cross-model fusion": "#59A14F",
    "Neural pair scorer": "#F28E2B",
    "Graph learning": "#B07AA1",
    "Final ensemble": "#E15759",
}

fig, ax = plt.subplots(figsize=(12, 8))

colors = plot_df["family"].map(family_palette).fillna("#999999")

ax.barh(
    y=np.arange(len(plot_df)),
    width=plot_df["pr_auc_mean"],
    color=colors,
    edgecolor="black",
    linewidth=0.7
)

ax.errorbar(
    plot_df["pr_auc_mean"],
    np.arange(len(plot_df)),
    xerr=plot_df["pr_auc_std"].fillna(0),
    fmt="none",
    ecolor="black",
    elinewidth=1.1,
    capsize=3
)

ax.set_yticks(np.arange(len(plot_df)))
ax.set_yticklabels(plot_df["short_label"], fontsize=11)

ax.set_xlabel("PR-AUC", fontsize=13, fontweight="bold")
ax.set_title(
    "Figure 4A. Performance comparison of representative model families",
    loc="left",
    fontsize=15,
    fontweight="bold"
)

xmin = max(0, plot_df["pr_auc_mean"].min() - 0.04)
xmax = min(1.0, plot_df["pr_auc_mean"].max() + 0.025)
ax.set_xlim(xmin, xmax)

for i, r in plot_df.reset_index(drop=True).iterrows():
    ax.text(
        r["pr_auc_mean"] + 0.004,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

legend_handles = [
    Line2D([0], [0], marker="s", color="w", label=k,
           markerfacecolor=v, markeredgecolor="black", markersize=11)
    for k, v in family_palette.items()
    if k in plot_df["family"].unique()
]

ax.legend(
    handles=legend_handles,
    title="Model family",
    loc="lower right",
    frameon=True,
    fontsize=10,
    title_fontsize=11
)

ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)

plt.tight_layout()

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4A_model_family_pr_auc_publication"
)

plt.show()

Cell 7 — Figure 4B: ROC-AUC و PR-AUC کنار هم، خواناتر از heatmap

In [ ]:
metric_plot = plot_df.sort_values("pr_auc_mean", ascending=False).copy()

x = np.arange(len(metric_plot))
width = 0.36

fig, ax = plt.subplots(figsize=(14, 7))

ax.bar(
    x - width/2,
    metric_plot["pr_auc_mean"],
    width,
    label="PR-AUC",
    color="#E15759",
    edgecolor="black",
    linewidth=0.7
)

ax.bar(
    x + width/2,
    metric_plot["roc_auc_mean"],
    width,
    label="ROC-AUC",
    color="#4E79A7",
    edgecolor="black",
    linewidth=0.7
)

ax.errorbar(
    x - width/2,
    metric_plot["pr_auc_mean"],
    yerr=metric_plot["pr_auc_std"].fillna(0),
    fmt="none",
    ecolor="black",
    elinewidth=1,
    capsize=3
)

ax.errorbar(
    x + width/2,
    metric_plot["roc_auc_mean"],
    yerr=metric_plot["roc_auc_std"].fillna(0),
    fmt="none",
    ecolor="black",
    elinewidth=1,
    capsize=3
)

ax.set_xticks(x)
ax.set_xticklabels(metric_plot["short_label"], rotation=0, ha="center", fontsize=10)
ax.set_ylim(0.55, 1.0)

ax.set_ylabel("Cross-validation score", fontsize=13, fontweight="bold")
ax.set_title(
    "Figure 4B. ROC-AUC and PR-AUC across representative models",
    loc="left",
    fontsize=15,
    fontweight="bold"
)

ax.legend(frameon=True, fontsize=11)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

plt.tight_layout()

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4B_roc_pr_auc_grouped_bar_publication"
)

plt.show()

Cell 8 — Figure 4C: Metric profile مدل نهایی، خوانا و مقاله‌ای

In [ ]:
best = all_models_clean.sort_values("pr_auc_mean", ascending=False).iloc[0]

metric_df = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "F1",
        "Accuracy",
        "Precision",
        "Recall"
    ],
    "Score": [
        best["roc_auc_mean"],
        best["pr_auc_mean"],
        best["f1_mean"],
        best["accuracy_mean"],
        best["precision_mean"],
        best["recall_mean"]
    ]
})

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(
    metric_df["Metric"],
    metric_df["Score"],
    color="#2F4B7C",
    edgecolor="black",
    linewidth=0.7
)

ax.set_ylim(0, 1.0)
ax.set_ylabel("Score", fontsize=13, fontweight="bold")
ax.set_title(
    f"Figure 4C. Final model metric profile\n{best['short_label'].replace(chr(10), ' ')}",
    loc="left",
    fontsize=15,
    fontweight="bold"
)

for b in bars:
    h = b.get_height()
    ax.text(
        b.get_x() + b.get_width()/2,
        h + 0.018,
        f"{h:.3f}",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

plt.tight_layout()

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4C_final_model_metric_profile_publication"
)

plt.show()

Cell 9 — به‌جای heatmap شلوغ: جدول تصویری Top Models

In [ ]:
table_df = (
    all_models_clean
    .sort_values("pr_auc_mean", ascending=False)
    .head(10)
    .copy()
)

table_show = table_df[
    [
        "short_label",
        "family",
        "roc_auc_mean",
        "pr_auc_mean",
        "f1_mean",
        "accuracy_mean",
        "brier_mean"
    ]
].copy()

table_show.columns = [
    "Model",
    "Family",
    "ROC-AUC",
    "PR-AUC",
    "F1",
    "Accuracy",
    "Brier"
]

for c in ["ROC-AUC", "PR-AUC", "F1", "Accuracy", "Brier"]:
    table_show[c] = table_show[c].map(lambda x: f"{x:.3f}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")

table = ax.table(
    cellText=table_show.values,
    colLabels=table_show.columns,
    cellLoc="center",
    colLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.9)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.5)

    if row == 0:
        cell.set_text_props(weight="bold", color="white")
        cell.set_facecolor("#2F4B7C")
    else:
        if row % 2 == 0:
            cell.set_facecolor("#F2F2F2")
        else:
            cell.set_facecolor("white")

ax.set_title(
    "Figure 4D. Top model comparison table",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=18
)

plt.tight_layout()

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4D_top_model_table_publication"
)

plt.show()

table_show.to_csv(
    DAY19_DIR / "figure4" / "figure4D_top_model_table.csv",
    index=False
)

قدم بعدی: اول باید جدول embedding benchmark را دقیق بسازیم

این Cell را اجرا کن و خروجی را بفرست:

In [ ]:
# Cell 10 — Embedding model benchmark extraction

embedding_bench = baseline.copy()

embedding_bench["model_family"] = embedding_bench["embedding_model"].astype(str)
embedding_bench["feature_type"] = embedding_bench["feature_type"].astype(str)
embedding_bench["algorithm"] = embedding_bench["algorithm"].astype(str)

display(
    embedding_bench[
        [
            "embedding_model",
            "feature_type",
            "algorithm",
            "roc_auc_mean",
            "roc_auc_std",
            "pr_auc_mean",
            "pr_auc_std",
            "f1_mean",
            "accuracy_mean",
            "precision_mean",
            "recall_mean",
            "brier_mean",
        ]
    ]
    .sort_values("pr_auc_mean", ascending=False)
    .head(40)
)

print(embedding_bench["embedding_model"].unique())
print(embedding_bench["feature_type"].unique())
print(embedding_bench["algorithm"].unique())

Cell 11 — تمیزسازی نام مدل‌ها و اضافه کردن تعداد پارامتر

In [ ]:
embedding_bench = baseline.copy()

param_map = {
    "esm2_t6_8M_UR50D": 8,
    "esm2_t12_35M_UR50D": 35,
    "esm2_t30_150M_UR50D": 150,
    "esm2_t33_650M_UR50D": 650,
    "esm2_t36_3B_UR50D": 3000,
    "prot_bert": 420,
    "prot_bert_bfd": 420,
}

label_map = {
    "esm2_t6_8M_UR50D": "ESM2-8M",
    "esm2_t12_35M_UR50D": "ESM2-35M",
    "esm2_t30_150M_UR50D": "ESM2-150M",
    "esm2_t33_650M_UR50D": "ESM2-650M",
    "esm2_t36_3B_UR50D": "ESM2-3B",
    "prot_bert": "ProtBERT",
    "prot_bert_bfd": "ProtBERT-BFD",
}

embedding_bench["model_label"] = embedding_bench["embedding_model"].map(label_map)
embedding_bench["params_million"] = embedding_bench["embedding_model"].map(param_map)

best_embedding = (
    embedding_bench
    .sort_values("pr_auc_mean", ascending=False)
    .groupby(["embedding_model", "model_label", "params_million"], as_index=False)
    .first()
    .sort_values("pr_auc_mean", ascending=False)
)

display(best_embedding[
    [
        "model_label",
        "params_million",
        "feature_type",
        "algorithm",
        "roc_auc_mean",
        "pr_auc_mean",
        "pr_auc_std",
        "f1_mean",
        "brier_mean",
    ]
])

Cell 12 — Figure 4E: بهترین عملکرد هر embedding model

In [ ]:
plot_df = best_embedding.copy()
plot_df = plot_df.sort_values("pr_auc_mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    plot_df["model_label"],
    plot_df["pr_auc_mean"],
    xerr=plot_df["pr_auc_std"],
    color="#4E79A7",
    edgecolor="black",
    linewidth=0.8,
    capsize=4,
)

ax.set_xlabel("Best PR-AUC across feature types and algorithms", fontsize=12, fontweight="bold")
ax.set_ylabel("")
ax.set_title(
    "Figure 4E. Protein language model benchmark",
    loc="left",
    fontsize=15,
    fontweight="bold",
)

xmin = plot_df["pr_auc_mean"].min() - 0.03
xmax = plot_df["pr_auc_mean"].max() + 0.02
ax.set_xlim(xmin, xmax)

for i, r in plot_df.reset_index(drop=True).iterrows():
    ax.text(
        r["pr_auc_mean"] + 0.004,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold",
    )

ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)

plt.tight_layout()

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4E_embedding_model_benchmark"
)

plt.show()

Cell 13 — Figure 4F: رابطه تعداد پارامتر و PR-AUC

In [ ]:
scaling_df = best_embedding.copy()

fig, ax = plt.subplots(figsize=(9, 6))

esm_df = scaling_df[scaling_df["embedding_model"].str.contains("esm2")].copy()
bert_df = scaling_df[scaling_df["embedding_model"].str.contains("prot_bert")].copy()

ax.plot(
    esm_df.sort_values("params_million")["params_million"],
    esm_df.sort_values("params_million")["pr_auc_mean"],
    marker="o",
    linewidth=2.5,
    markersize=8,
    label="ESM2 family",
    color="#E15759",
)

ax.scatter(
    bert_df["params_million"],
    bert_df["pr_auc_mean"],
    s=100,
    color="#4E79A7",
    edgecolor="black",
    linewidth=0.8,
    label="ProtBERT family",
)

for _, r in scaling_df.iterrows():
    ax.text(
        r["params_million"] * 1.04,
        r["pr_auc_mean"],
        r["model_label"],
        fontsize=9,
        va="center",
    )

ax.set_xscale("log")
ax.set_xlabel("Model size, million parameters log scale", fontsize=12, fontweight="bold")
ax.set_ylabel("Best PR-AUC", fontsize=12, fontweight="bold")
ax.set_title(
    "Figure 4F. Parameter scaling of protein language models",
    loc="left",
    fontsize=15,
    fontweight="bold",
)

ax.set_ylim(scaling_df["pr_auc_mean"].min() - 0.02, scaling_df["pr_auc_mean"].max() + 0.02)
ax.legend(frameon=True)
ax.grid(True, alpha=0.25)

plt.tight_layout()

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4F_parameter_scaling_pr_auc"
)

plt.show()

Cell 14 — Figure 4G: per-sequence / per-residue / fusion comparison

In [ ]:
feature_best = (
    embedding_bench
    .sort_values("pr_auc_mean", ascending=False)
    .groupby(["model_label", "feature_type"], as_index=False)
    .first()
)

feature_order = ["per_sequence", "per_residue", "fusion"]

model_order = [
    "ESM2-8M",
    "ESM2-35M",
    "ESM2-150M",
    "ESM2-650M",
    "ESM2-3B",
    "ProtBERT",
    "ProtBERT-BFD",
]

feature_best["model_label"] = pd.Categorical(
    feature_best["model_label"],
    categories=model_order,
    ordered=True,
)

feature_best["feature_type"] = pd.Categorical(
    feature_best["feature_type"],
    categories=feature_order,
    ordered=True,
)

feature_best = feature_best.sort_values(["model_label", "feature_type"])

fig, ax = plt.subplots(figsize=(13, 6))

sns.barplot(
    data=feature_best,
    x="model_label",
    y="pr_auc_mean",
    hue="feature_type",
    palette=["#4E79A7", "#F28E2B", "#59A14F"],
    edgecolor="black",
    linewidth=0.6,
    ax=ax,
)

ax.set_ylim(0.60, 0.88)
ax.set_xlabel("")
ax.set_ylabel("Best PR-AUC", fontsize=12, fontweight="bold")
ax.set_title(
    "Figure 4G. Feature representation benchmark across protein language models",
    loc="left",
    fontsize=15,
    fontweight="bold",
)

ax.legend(
    title="Feature type",
    frameon=True,
    fontsize=10,
    title_fontsize=11,
)

ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

plt.tight_layout()

save_pubfig(
    fig,
    DAY19_DIR / "figure4",
    "figure4G_feature_type_embedding_benchmark"
)

plt.show()

Cell 15 — جدول خلاصه برای گزارش مقاله

In [ ]:
embedding_summary_for_paper = best_embedding[
    [
        "model_label",
        "params_million",
        "feature_type",
        "algorithm",
        "roc_auc_mean",
        "roc_auc_std",
        "pr_auc_mean",
        "pr_auc_std",
        "f1_mean",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "brier_mean",
    ]
].copy()

embedding_summary_for_paper = embedding_summary_for_paper.sort_values(
    "pr_auc_mean",
    ascending=False
)

display(embedding_summary_for_paper)

embedding_summary_for_paper.to_csv(
    DAY19_DIR / "figure4" / "embedding_model_benchmark_summary_for_paper.csv",
    index=False
)

Cell 1 — نصب پکیج‌های لازم

In [ ]:
!pip install scienceplots adjustText statannotations svgutils openpyxl -q

Cell 2 — Setup حرفه‌ای Figureها

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import scienceplots
from adjustText import adjust_text

from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
from matplotlib.lines import Line2D

PROJECT_ROOT = Path(".")

DATA_PROC = PROJECT_ROOT / "Data_proc"
DATA_ML = PROJECT_ROOT / "Data_ml"
GRAPH_DIR = DATA_ML / "graph_dataset"

DAY13_DIR = GRAPH_DIR / "day13_explainability"
DAY14_DIR = GRAPH_DIR / "day14_case_studies"
DAY15_DIR = GRAPH_DIR / "day15_external_validation"
DAY16_DIR = GRAPH_DIR / "day16_novel_predictions"
DAY17_DIR = GRAPH_DIR / "day17_novel_explainability"
DAY18_DIR = GRAPH_DIR / "day18_cancer_specific_analysis"

DAY19_DIR = GRAPH_DIR / "day19_paper_figures_final"
DAY19_DIR.mkdir(parents=True, exist_ok=True)

for i in range(1, 11):
    (DAY19_DIR / f"Figure_{i}").mkdir(parents=True, exist_ok=True)

(DAY19_DIR / "tables").mkdir(exist_ok=True)
(DAY19_DIR / "metadata").mkdir(exist_ok=True)

plt.style.use(["science", "no-latex"])

mpl.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 700,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "Arial",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 1.1,
})

PALETTE = {
    "blue": "#2F5597",
    "red": "#C00000",
    "orange": "#ED7D31",
    "green": "#70AD47",
    "purple": "#7030A0",
    "gray": "#7F7F7F",
    "lightgray": "#E7E6E6",
    "dark": "#222222",
}

print("Day 19 final figure directory:")
print(DAY19_DIR)

Cell 3 — تابع ذخیره Publication-grade

In [ ]:
def save_final_figure(fig, fig_number, filename):
    outdir = DAY19_DIR / f"Figure_{fig_number}"
    outdir.mkdir(parents=True, exist_ok=True)

    for ext in ["png", "pdf", "svg"]:
        path = outdir / f"{filename}.{ext}"
        fig.savefig(path, bbox_inches="tight", dpi=700 if ext == "png" else None)
        print("saved:", path)

Cell 4 — لود همه فایل‌های کلیدی

In [ ]:
data = {}

data["baseline"] = pd.read_csv(DATA_ML / "baseline_results" / "baseline_results_all.csv")
data["leaderboard"] = pd.read_csv(DATA_ML / "baseline_results" / "leaderboard_pr_auc.csv")
data["cross_model"] = pd.read_csv(DATA_ML / "cross_model_fusion_results" / "cross_model_fusion_results_all.csv")
data["nn_compare"] = pd.read_csv(DATA_ML / "neural_pair_scorer_results" / "nn_all_losses_final_comparison.csv")
data["graph_leaderboard"] = pd.read_csv(GRAPH_DIR / "day12_graph_final_leaderboard.csv")
data["mega"] = pd.read_csv(GRAPH_DIR / "mega_weighted_ensemble_results.csv")

data["graph_nodes"] = pd.read_csv(GRAPH_DIR / "graph_nodes.csv")
data["interaction_edges"] = pd.read_csv(GRAPH_DIR / "interaction_edges_labeled.csv")
data["ppi_edges"] = pd.read_csv(GRAPH_DIR / "ppi_edges.csv")
data["loc_edges"] = pd.read_csv(GRAPH_DIR / "colocalization_edges.csv")

data["edge_importance"] = pd.read_csv(DAY13_DIR / "edge_type_importance_from_ablation.csv")
data["top_prediction_enrichment"] = pd.read_csv(DAY13_DIR / "top_prediction_enrichment.csv")

data["case_study"] = pd.read_csv(DAY14_DIR / "case_study_article_table.csv")
data["external_validation"] = pd.read_csv(DAY15_DIR / "uniprot_curated_validation_cases.csv")

data["novel_top500"] = pd.read_csv(DAY16_DIR / "top500_novel_predictions_hgt.csv")
data["novel_qc"] = pd.read_csv(DAY16_DIR / "day16_novel_prediction_qc.csv")

data["day17_priority"] = pd.read_csv(DAY17_DIR / "day17_final_novel_candidate_priority_table.csv")
data["relation_importance"] = pd.read_csv(DAY17_DIR / "novel_candidate_relation_importance_percent.csv")

data["day18_priority"] = pd.read_csv(DAY18_DIR / "day18_final_gi_cancer_priority_candidates.csv")
data["driver_summary"] = pd.read_csv(DAY18_DIR / "driver_enrichment_summary.csv")
data["pathway_freq"] = pd.read_csv(DAY18_DIR / "gi_driver_pathway_frequency.csv")
data["gi_rank"] = pd.read_csv(DAY18_DIR / "gi_enzyme_rank.csv")

for k, df in data.items():
    print(k, df.shape)

Cell 5 — ساخت جدول unified برای همه مدل‌ها

In [ ]:
def add_std_if_missing(df):
    df = df.copy()
    for col in ["roc_auc_std", "pr_auc_std", "f1_std", "accuracy_std", "precision_std", "recall_std", "brier_std"]:
        if col not in df.columns:
            df[col] = np.nan
    return df

rows = []

baseline = add_std_if_missing(data["baseline"])
baseline["family"] = "Pair-level ML"
baseline["model_name"] = (
    baseline["embedding_model"].astype(str) + " | " +
    baseline["feature_type"].astype(str) + " | " +
    baseline["algorithm"].astype(str)
)
rows.append(baseline)

cross_model = add_std_if_missing(data["cross_model"])
cross_model["family"] = "Cross-model fusion"
cross_model["model_name"] = cross_model["fusion_name"].astype(str) + " | " + cross_model["algorithm"].astype(str)
rows.append(cross_model)

nn = add_std_if_missing(data["nn_compare"])
nn["family"] = "Neural pair scorer"
nn["model_name"] = nn["model"].astype(str)
rows.append(nn)

graph = add_std_if_missing(data["graph_leaderboard"])
graph["family"] = "Graph learning"
graph["model_name"] = graph["model"].astype(str)
rows.append(graph)

mega = add_std_if_missing(data["mega"])
mega["family"] = "Final ensemble"
mega["model_name"] = mega["model"].astype(str)
rows.append(mega)

cols = [
    "family", "model_name",
    "roc_auc_mean", "roc_auc_std",
    "pr_auc_mean", "pr_auc_std",
    "f1_mean", "f1_std",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "brier_mean", "brier_std",
]

all_models = pd.concat([r[cols] for r in rows], ignore_index=True)
all_models = all_models.sort_values("pr_auc_mean", ascending=False).reset_index(drop=True)
all_models["rank"] = np.arange(1, len(all_models) + 1)

all_models.to_csv(DAY19_DIR / "tables" / "all_models_unified_benchmark.csv", index=False)

display(all_models.head(20))

Cell 6 — آماده‌سازی Figure 2 data

In [ ]:
embedding_bench = data["baseline"].copy()

param_map = {
    "esm2_t6_8M_UR50D": 8,
    "esm2_t12_35M_UR50D": 35,
    "esm2_t30_150M_UR50D": 150,
    "esm2_t33_650M_UR50D": 650,
    "esm2_t36_3B_UR50D": 3000,
    "prot_bert": 420,
    "prot_bert_bfd": 420,
}

label_map = {
    "esm2_t6_8M_UR50D": "ESM2-8M",
    "esm2_t12_35M_UR50D": "ESM2-35M",
    "esm2_t30_150M_UR50D": "ESM2-150M",
    "esm2_t33_650M_UR50D": "ESM2-650M",
    "esm2_t36_3B_UR50D": "ESM2-3B",
    "prot_bert": "ProtBERT",
    "prot_bert_bfd": "ProtBERT-BFD",
}

embedding_bench["model_label"] = embedding_bench["embedding_model"].map(label_map)
embedding_bench["params_million"] = embedding_bench["embedding_model"].map(param_map)

best_embedding = (
    embedding_bench
    .sort_values("pr_auc_mean", ascending=False)
    .groupby(["embedding_model", "model_label", "params_million"], as_index=False)
    .first()
    .sort_values("params_million")
)

feature_best = (
    embedding_bench
    .sort_values("pr_auc_mean", ascending=False)
    .groupby(["model_label", "feature_type"], as_index=False)
    .first()
)

model_order = [
    "ESM2-8M",
    "ESM2-35M",
    "ESM2-150M",
    "ProtBERT",
    "ProtBERT-BFD",
    "ESM2-650M",
    "ESM2-3B",
]

feature_order = ["per_sequence", "per_residue", "fusion"]

best_embedding["model_label"] = pd.Categorical(
    best_embedding["model_label"],
    categories=model_order,
    ordered=True
)

feature_best["model_label"] = pd.Categorical(
    feature_best["model_label"],
    categories=model_order,
    ordered=True
)

feature_best["feature_type"] = pd.Categorical(
    feature_best["feature_type"],
    categories=feature_order,
    ordered=True
)

best_embedding = best_embedding.sort_values("model_label")
feature_best = feature_best.sort_values(["model_label", "feature_type"])

best_embedding.to_csv(
    DAY19_DIR / "Figure_2" / "figure2_best_embedding_models.csv",
    index=False
)

feature_best.to_csv(
    DAY19_DIR / "Figure_2" / "figure2_feature_type_embedding_benchmark.csv",
    index=False
)

display(best_embedding)
display(feature_best.head())

Cell 7 — Figure 2 نهایی multi-panel

In [ ]:
fig = plt.figure(figsize=(16, 12))

gs = GridSpec(
    2, 2,
    figure=fig,
    height_ratios=[1.05, 1],
    width_ratios=[1.1, 1],
    hspace=0.32,
    wspace=0.28
)

# -------------------------
# Panel A
# -------------------------
axA = fig.add_subplot(gs[0, 0])

plotA = best_embedding.sort_values("pr_auc_mean", ascending=True)

axA.barh(
    plotA["model_label"].astype(str),
    plotA["pr_auc_mean"],
    xerr=plotA["pr_auc_std"],
    color="#2F5597",
    edgecolor="black",
    linewidth=0.7,
    capsize=3
)

axA.set_xlabel("Best PR-AUC")
axA.set_ylabel("")
axA.set_title("A. Best performance by protein language model", loc="left", fontweight="bold")

axA.set_xlim(
    plotA["pr_auc_mean"].min() - 0.025,
    plotA["pr_auc_mean"].max() + 0.02
)

for i, r in plotA.reset_index(drop=True).iterrows():
    axA.text(
        r["pr_auc_mean"] + 0.003,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=9,
        fontweight="bold"
    )

axA.grid(axis="x", alpha=0.25)
axA.grid(axis="y", visible=False)

# -------------------------
# Panel B
# -------------------------
axB = fig.add_subplot(gs[0, 1])

scaling = best_embedding.copy()
esm = scaling[scaling["embedding_model"].str.contains("esm2")].sort_values("params_million")
bert = scaling[scaling["embedding_model"].str.contains("prot_bert")]

axB.plot(
    esm["params_million"],
    esm["pr_auc_mean"],
    marker="o",
    markersize=7,
    linewidth=2.4,
    color="#C00000",
    label="ESM2 family"
)

axB.scatter(
    bert["params_million"],
    bert["pr_auc_mean"],
    s=90,
    color="#2F5597",
    edgecolor="black",
    linewidth=0.8,
    label="ProtBERT family"
)

texts = []
for _, r in scaling.iterrows():
    texts.append(
        axB.text(
            r["params_million"],
            r["pr_auc_mean"],
            str(r["model_label"]),
            fontsize=8
        )
    )

adjust_text(
    texts,
    ax=axB,
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.5)
)

axB.set_xscale("log")
axB.set_xlabel("Model size, million parameters log scale")
axB.set_ylabel("Best PR-AUC")
axB.set_title("B. Parameter scaling and performance saturation", loc="left", fontweight="bold")
axB.legend(frameon=True)
axB.grid(True, alpha=0.25)

# -------------------------
# Panel C
# -------------------------
axC = fig.add_subplot(gs[1, 0])

feature_colors = {
    "per_sequence": "#2F5597",
    "per_residue": "#ED7D31",
    "fusion": "#70AD47",
}

sns.barplot(
    data=feature_best,
    x="model_label",
    y="pr_auc_mean",
    hue="feature_type",
    palette=feature_colors,
    edgecolor="black",
    linewidth=0.5,
    ax=axC
)

axC.set_ylim(0.60, 0.88)
axC.set_xlabel("")
axC.set_ylabel("Best PR-AUC")
axC.set_title("C. Feature representation benchmark", loc="left", fontweight="bold")
axC.tick_params(axis="x", rotation=25)
axC.legend(title="Feature type", frameon=True)
axC.grid(axis="y", alpha=0.25)
axC.grid(axis="x", visible=False)

# -------------------------
# Panel D
# -------------------------
axD = fig.add_subplot(gs[1, 1])
axD.axis("off")

table_df = best_embedding.copy()
table_df = table_df.sort_values("pr_auc_mean", ascending=False)

table_show = table_df[
    [
        "model_label",
        "params_million",
        "feature_type",
        "algorithm",
        "roc_auc_mean",
        "pr_auc_mean",
        "brier_mean",
    ]
].copy()

table_show.columns = [
    "Model",
    "Params M",
    "Best feature",
    "Algorithm",
    "ROC-AUC",
    "PR-AUC",
    "Brier",
]

for c in ["ROC-AUC", "PR-AUC", "Brier"]:
    table_show[c] = table_show[c].map(lambda x: f"{x:.3f}")

table_show["Params M"] = table_show["Params M"].map(lambda x: f"{int(x)}")

tbl = axD.table(
    cellText=table_show.values,
    colLabels=table_show.columns,
    cellLoc="center",
    colLoc="center",
    loc="center"
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1.05, 1.55)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.4)

    if row == 0:
        cell.set_facecolor("#2F5597")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F7F7F7" if row % 2 == 0 else "white")

axD.set_title("D. Best configuration per embedding model", loc="left", fontweight="bold", pad=12)

fig.suptitle(
    "Figure 2. Benchmarking protein language models for E3/DUB specificity prediction",
    fontsize=16,
    fontweight="bold",
    y=0.995
)

save_final_figure(
    fig,
    2,
    "Figure2_protein_language_model_benchmark"
)

plt.show()

In [ ]:
fig = plt.figure(figsize=(20, 16), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.15, 1.15, 0.95],
    width_ratios=[1, 1],
    hspace=0.42,
    wspace=0.35
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, :])
axD = fig.add_subplot(gs[2, :])

# Panel A
plotA = best_embedding.sort_values("pr_auc_mean", ascending=True)

axA.barh(
    plotA["model_label"].astype(str),
    plotA["pr_auc_mean"],
    xerr=plotA["pr_auc_std"],
    color="#2F5597",
    edgecolor="black",
    linewidth=0.8,
    capsize=4
)

axA.set_xlabel("Best PR-AUC")
axA.set_ylabel("")
axA.set_title("A. Best performance by protein language model", loc="left", fontweight="bold")
axA.set_xlim(plotA["pr_auc_mean"].min() - 0.03, plotA["pr_auc_mean"].max() + 0.03)

for i, r in plotA.reset_index(drop=True).iterrows():
    axA.text(r["pr_auc_mean"] + 0.004, i, f"{r['pr_auc_mean']:.3f}",
             va="center", fontsize=10, fontweight="bold")

axA.grid(axis="x", alpha=0.25)
axA.grid(axis="y", visible=False)

# Panel B
scaling = best_embedding.copy()
esm = scaling[scaling["embedding_model"].str.contains("esm2")].sort_values("params_million")
bert = scaling[scaling["embedding_model"].str.contains("prot_bert")]

axB.plot(
    esm["params_million"], esm["pr_auc_mean"],
    marker="o", markersize=8, linewidth=2.6,
    color="#C00000", label="ESM2 family"
)

axB.scatter(
    bert["params_million"], bert["pr_auc_mean"],
    s=110, color="#2F5597", edgecolor="black",
    linewidth=0.9, label="ProtBERT family"
)

label_offsets = {
    "ESM2-8M": (1.08, -0.004),
    "ESM2-35M": (1.08, -0.003),
    "ESM2-150M": (1.08, 0.001),
    "ESM2-650M": (1.08, 0.003),
    "ESM2-3B": (0.82, 0.004),
    "ProtBERT": (0.70, -0.010),
    "ProtBERT-BFD": (1.12, 0.010),
}

for _, r in scaling.iterrows():
    mult, dy = label_offsets.get(str(r["model_label"]), (1.05, 0))
    axB.text(
        r["params_million"] * mult,
        r["pr_auc_mean"] + dy,
        str(r["model_label"]),
        fontsize=9,
        fontweight="bold"
    )

axB.set_xscale("log")
axB.set_xlabel("Model size, million parameters log scale")
axB.set_ylabel("Best PR-AUC")
axB.set_title("B. Parameter scaling and performance saturation", loc="left", fontweight="bold")
axB.set_ylim(scaling["pr_auc_mean"].min() - 0.025, scaling["pr_auc_mean"].max() + 0.025)
axB.legend(frameon=True, loc="lower right")
axB.grid(True, alpha=0.25)

# Panel C
feature_colors = {
    "per_sequence": "#2F5597",
    "per_residue": "#ED7D31",
    "fusion": "#70AD47",
}

sns.barplot(
    data=feature_best,
    x="model_label",
    y="pr_auc_mean",
    hue="feature_type",
    palette=feature_colors,
    edgecolor="black",
    linewidth=0.6,
    ax=axC
)

axC.set_ylim(0.60, 0.88)
axC.set_xlabel("")
axC.set_ylabel("Best PR-AUC")
axC.set_title("C. Feature representation benchmark", loc="left", fontweight="bold")
axC.tick_params(axis="x", rotation=0)
axC.legend(title="Feature type", frameon=True, loc="upper left", ncol=3)
axC.grid(axis="y", alpha=0.25)
axC.grid(axis="x", visible=False)

# Panel D
axD.axis("off")

table_df = best_embedding.sort_values("pr_auc_mean", ascending=False).copy()

table_show = table_df[
    ["model_label", "params_million", "feature_type", "algorithm",
     "roc_auc_mean", "pr_auc_mean", "brier_mean"]
].copy()

table_show.columns = ["Model", "Params M", "Best feature", "Algorithm", "ROC-AUC", "PR-AUC", "Brier"]

for c in ["ROC-AUC", "PR-AUC", "Brier"]:
    table_show[c] = table_show[c].map(lambda x: f"{x:.3f}")

table_show["Params M"] = table_show["Params M"].map(lambda x: f"{int(x)}")

tbl = axD.table(
    cellText=table_show.values,
    colLabels=table_show.columns,
    cellLoc="center",
    colLoc="center",
    loc="center"
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.15, 1.8)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor("#2F5597")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F7F7F7" if row % 2 == 0 else "white")

axD.set_title("D. Best configuration per embedding model", loc="left", fontweight="bold", pad=20)

fig.suptitle(
    "Figure 2. Benchmarking protein language models for E3/DUB specificity prediction",
    fontsize=18,
    fontweight="bold",
    y=0.985
)

plt.subplots_adjust(top=0.93, bottom=0.04, left=0.08, right=0.98)

save_final_figure(
    fig,
    2,
    "Figure2_protein_language_model_benchmark_v2_spacious"
)

plt.show()

Figure 1 — کد نهایی

In [ ]:
# Figure 1 — Overall E3/DUB project workflow

fig = plt.figure(figsize=(20, 15), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.15, 1.15, 0.9],
    width_ratios=[1, 1],
    hspace=0.42,
    wspace=0.32
)

axA = fig.add_subplot(gs[0, :])
axB = fig.add_subplot(gs[1, 0])
axC = fig.add_subplot(gs[1, 1])
axD = fig.add_subplot(gs[2, :])

for ax in [axA, axB, axC, axD]:
    ax.axis("off")

# -------------------------
# Panel A: main workflow
# -------------------------
workflow_steps = [
    ("Raw databases", "UbiBrowser\nBioGRID\nUniProt\nTISCH2"),
    ("Data curation", "Positive pairs\nNegative sampling\nContext filters"),
    ("Protein representation", "ESM2 / ProtBERT\nPer-sequence\nPer-residue"),
    ("Pair-level models", "Logistic\nRandom forest\nXGBoost\nNeural scorer"),
    ("Graph learning", "GraphSAGE\nGAT\nHeteroGNN\nHGT"),
    ("Discovery", "External validation\nNovel prediction\nExplainability\nCancer prioritization"),
]

x_positions = np.linspace(0.08, 0.92, len(workflow_steps))
y = 0.53

for i, (title, text) in enumerate(workflow_steps):
    x = x_positions[i]

    box = FancyBboxPatch(
        (x - 0.068, y - 0.18),
        0.136,
        0.36,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        linewidth=1.4,
        edgecolor="#222222",
        facecolor="#F7F9FC",
        transform=axA.transAxes,
    )
    axA.add_patch(box)

    axA.text(
        x, y + 0.095,
        title,
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold",
        transform=axA.transAxes,
    )

    axA.text(
        x, y - 0.045,
        text,
        ha="center",
        va="center",
        fontsize=10,
        linespacing=1.35,
        transform=axA.transAxes,
    )

    if i < len(workflow_steps) - 1:
        arrow = FancyArrowPatch(
            (x + 0.078, y),
            (x_positions[i + 1] - 0.078, y),
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color="#333333",
            transform=axA.transAxes,
        )
        axA.add_patch(arrow)

axA.set_title(
    "A. End-to-end computational workflow",
    loc="left",
    fontsize=14,
    fontweight="bold",
)

# -------------------------
# Panel B: data sources
# -------------------------
source_items = [
    ("UbiBrowser", "E3/DUB-substrate\npositive interactions"),
    ("BioGRID", "physical PPI\nnetwork context"),
    ("UniProt", "subcellular localization\nprotein sequences"),
    ("TISCH2", "CRC and LIHC\nsingle-cell expression"),
]

for i, (name, desc) in enumerate(source_items):
    yy = 0.82 - i * 0.22

    circ = Circle(
        (0.12, yy),
        0.055,
        facecolor="#2F5597",
        edgecolor="black",
        linewidth=0.8,
        transform=axB.transAxes,
    )
    axB.add_patch(circ)

    axB.text(
        0.12, yy,
        str(i + 1),
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        color="white",
        transform=axB.transAxes,
    )

    axB.text(
        0.22, yy + 0.035,
        name,
        ha="left",
        va="center",
        fontsize=12,
        fontweight="bold",
        transform=axB.transAxes,
    )

    axB.text(
        0.22, yy - 0.045,
        desc,
        ha="left",
        va="center",
        fontsize=10,
        linespacing=1.25,
        transform=axB.transAxes,
    )

axB.set_title(
    "B. Biological data sources",
    loc="left",
    fontsize=14,
    fontweight="bold",
)

# -------------------------
# Panel C: graph construction summary
# -------------------------
summary_items = [
    ("Proteins", "2,105"),
    ("Labeled pairs", "6,196"),
    ("PPI edges", "60,723"),
    ("Co-localization edges", "193,159"),
    ("Total graph edges", "260,078"),
]

for i, (label, value) in enumerate(summary_items):
    yy = 0.82 - i * 0.17

    box = FancyBboxPatch(
        (0.08, yy - 0.055),
        0.84,
        0.10,
        boxstyle="round,pad=0.012,rounding_size=0.02",
        linewidth=1.0,
        edgecolor="#333333",
        facecolor="#FFFFFF",
        transform=axC.transAxes,
    )
    axC.add_patch(box)

    axC.text(
        0.14, yy,
        label,
        ha="left",
        va="center",
        fontsize=11,
        fontweight="bold",
        transform=axC.transAxes,
    )

    axC.text(
        0.86, yy,
        value,
        ha="right",
        va="center",
        fontsize=12,
        fontweight="bold",
        color="#C00000",
        transform=axC.transAxes,
    )

axC.set_title(
    "C. Final heterogeneous graph",
    loc="left",
    fontsize=14,
    fontweight="bold",
)

# -------------------------
# Panel D: model-to-discovery story
# -------------------------
story_steps = [
    ("Pair features", "Embedding-derived pair vectors"),
    ("Model selection", "Baseline, fusion, neural and graph models"),
    ("Best model", "MegaWeighted HGT ensemble"),
    ("Biological interpretation", "Explainability and relation dependency"),
    ("Cancer prioritization", "CRC, LIHC and GI-specific candidates"),
]

x_positions = np.linspace(0.07, 0.93, len(story_steps))
y = 0.55

for i, (title, text) in enumerate(story_steps):
    x = x_positions[i]

    box = FancyBboxPatch(
        (x - 0.075, y - 0.14),
        0.15,
        0.28,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        linewidth=1.3,
        edgecolor="#222222",
        facecolor="#F2F2F2",
        transform=axD.transAxes,
    )
    axD.add_patch(box)

    axD.text(
        x, y + 0.06,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        transform=axD.transAxes,
    )

    axD.text(
        x, y - 0.055,
        text,
        ha="center",
        va="center",
        fontsize=9.5,
        linespacing=1.25,
        transform=axD.transAxes,
    )

    if i < len(story_steps) - 1:
        arrow = FancyArrowPatch(
            (x + 0.087, y),
            (x_positions[i + 1] - 0.087, y),
            arrowstyle="-|>",
            mutation_scale=15,
            linewidth=1.7,
            color="#333333",
            transform=axD.transAxes,
        )
        axD.add_patch(arrow)

axD.set_title(
    "D. Model-to-discovery framework",
    loc="left",
    fontsize=14,
    fontweight="bold",
)

fig.suptitle(
    "Figure 1. Overview of the E3/DUB specificity prediction and discovery framework",
    fontsize=18,
    fontweight="bold",
    y=0.985,
)

plt.subplots_adjust(top=0.92, bottom=0.04, left=0.04, right=0.98)

save_final_figure(
    fig,
    1,
    "Figure1_E3DUB_overall_workflow"
)

plt.show()

بریم Figure 3: Dataset Construction and QC.

این Figure باید نشان بدهد دیتاست چطور از positive/negative خام به دیتاست نهایی مدل‌پذیر رسید.

Figure 3 — Dataset construction and QC

Cell 8 — آماده‌سازی داده‌های Figure 3

In [ ]:
# Figure 3 data preparation

pairs_all = pd.read_csv(DATA_PROC / "pairs" / "pairs_all_embedding_ready.csv")
final_pairs_qc = pd.read_csv(DATA_PROC / "qc_reports" / "final_pairs_qc.csv")
localization_qc = pd.read_csv(DATA_PROC / "qc_reports" / "localization_filter_qc.csv")
scrna_qc = pd.read_csv(DATA_PROC / "qc_reports" / "scrna_filter_qc.csv")

negative_e3_initial = pd.read_csv(DATA_PROC / "negatives" / "negative_e3_initial_2x.csv")
negative_dub_initial = pd.read_csv(DATA_PROC / "negatives" / "negative_dub_initial_2x.csv")

negative_e3_ppi = pd.read_csv(DATA_PROC / "negatives" / "negative_e3_after_ppi_filter.csv")
negative_dub_ppi = pd.read_csv(DATA_PROC / "negatives" / "negative_dub_after_ppi_filter.csv")

negative_e3_loc = pd.read_csv(DATA_PROC / "negatives" / "negative_e3_after_ppi_loc_filter.csv")
negative_dub_loc = pd.read_csv(DATA_PROC / "negatives" / "negative_dub_after_ppi_loc_filter.csv")

negative_e3_scrna = pd.read_csv(DATA_PROC / "negatives" / "negative_e3_after_ppi_loc_scrna_filter.csv")
negative_dub_scrna = pd.read_csv(DATA_PROC / "negatives" / "negative_dub_after_ppi_loc_scrna_filter.csv")

positive_e3 = pd.read_csv(DATA_PROC / "positives" / "positive_e3.csv")
positive_dub = pd.read_csv(DATA_PROC / "positives" / "positive_dub.csv")

negative_steps = pd.DataFrame([
    {
        "step": "Initial\n2x negatives",
        "E3": len(negative_e3_initial),
        "DUB": len(negative_dub_initial),
    },
    {
        "step": "After PPI\nfilter",
        "E3": len(negative_e3_ppi),
        "DUB": len(negative_dub_ppi),
    },
    {
        "step": "After localization\nfilter",
        "E3": len(negative_e3_loc),
        "DUB": len(negative_dub_loc),
    },
    {
        "step": "After scRNA\nfilter",
        "E3": len(negative_e3_scrna),
        "DUB": len(negative_dub_scrna),
    },
])

final_label_counts = (
    pairs_all["label"]
    .astype(int)
    .value_counts()
    .rename(index={0: "Negative", 1: "Positive"})
    .reset_index()
)

final_label_counts.columns = ["label", "count"]

class_counts = pd.DataFrame([
    {"class": "E3 positives", "count": len(positive_e3)},
    {"class": "DUB positives", "count": len(positive_dub)},
    {"class": "E3 negatives", "count": len(negative_e3_scrna)},
    {"class": "DUB negatives", "count": len(negative_dub_scrna)},
])

negative_steps.to_csv(DAY19_DIR / "Figure_3" / "figure3_negative_filter_steps.csv", index=False)
final_label_counts.to_csv(DAY19_DIR / "Figure_3" / "figure3_final_label_counts.csv", index=False)
class_counts.to_csv(DAY19_DIR / "Figure_3" / "figure3_class_counts.csv", index=False)

display(negative_steps)
display(final_label_counts)
display(class_counts)

Cell 9 — Figure 3 نهایی multi-panel

In [ ]:
# Figure 3 — Dataset construction and QC

fig = plt.figure(figsize=(20, 15), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.1, 1.15, 0.9],
    width_ratios=[1, 1],
    hspace=0.42,
    wspace=0.32
)

axA = fig.add_subplot(gs[0, :])
axB = fig.add_subplot(gs[1, 0])
axC = fig.add_subplot(gs[1, 1])
axD = fig.add_subplot(gs[2, :])

# -------------------------
# Panel A: dataset construction flow
# -------------------------
axA.axis("off")

flow_steps = [
    ("UbiBrowser\npositives", f"E3: {len(positive_e3):,}\nDUB: {len(positive_dub):,}"),
    ("Candidate\nnegatives", f"E3: {len(negative_e3_initial):,}\nDUB: {len(negative_dub_initial):,}"),
    ("PPI filter", f"E3: {len(negative_e3_ppi):,}\nDUB: {len(negative_dub_ppi):,}"),
    ("Localization\nfilter", f"E3: {len(negative_e3_loc):,}\nDUB: {len(negative_dub_loc):,}"),
    ("scRNA\nfilter", f"E3: {len(negative_e3_scrna):,}\nDUB: {len(negative_dub_scrna):,}"),
    ("Final model-ready\npairs", f"Total: {len(pairs_all):,}\nPositive/Negative"),
]

x_positions = np.linspace(0.07, 0.93, len(flow_steps))
y = 0.53

for i, (title, value) in enumerate(flow_steps):
    x = x_positions[i]

    box = FancyBboxPatch(
        (x - 0.065, y - 0.18),
        0.13,
        0.36,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        linewidth=1.3,
        edgecolor="#222222",
        facecolor="#F7F9FC",
        transform=axA.transAxes,
    )
    axA.add_patch(box)

    axA.text(
        x,
        y + 0.075,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        transform=axA.transAxes,
    )

    axA.text(
        x,
        y - 0.07,
        value,
        ha="center",
        va="center",
        fontsize=10,
        linespacing=1.25,
        color="#C00000" if i >= 2 else "#2F5597",
        fontweight="bold",
        transform=axA.transAxes,
    )

    if i < len(flow_steps) - 1:
        arrow = FancyArrowPatch(
            (x + 0.075, y),
            (x_positions[i + 1] - 0.075, y),
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color="#333333",
            transform=axA.transAxes,
        )
        axA.add_patch(arrow)

axA.set_title(
    "A. Dataset construction and context-aware negative filtering",
    loc="left",
    fontsize=14,
    fontweight="bold",
)

# -------------------------
# Panel B: negative reduction
# -------------------------
plot_neg = negative_steps.melt(
    id_vars="step",
    value_vars=["E3", "DUB"],
    var_name="enzyme_class",
    value_name="count",
)

sns.lineplot(
    data=plot_neg,
    x="step",
    y="count",
    hue="enzyme_class",
    marker="o",
    linewidth=2.8,
    markersize=8,
    palette={"E3": "#2F5597", "DUB": "#C00000"},
    ax=axB,
)

axB.set_title("B. Negative set reduction across biological filters", loc="left", fontweight="bold")
axB.set_xlabel("")
axB.set_ylabel("Number of candidate negatives")
axB.tick_params(axis="x", rotation=0)
axB.legend(title="Class", frameon=True)
axB.grid(axis="y", alpha=0.25)
axB.grid(axis="x", visible=False)

for _, r in plot_neg.iterrows():
    axB.text(
        r["step"],
        r["count"] + max(plot_neg["count"]) * 0.025,
        f"{int(r['count']):,}",
        ha="center",
        fontsize=8,
        fontweight="bold",
    )

# -------------------------
# Panel C: final class composition
# -------------------------
colors = ["#2F5597", "#70AD47", "#C00000", "#ED7D31"]

axC.bar(
    class_counts["class"],
    class_counts["count"],
    color=colors,
    edgecolor="black",
    linewidth=0.8,
)

axC.set_title("C. Final class composition", loc="left", fontweight="bold")
axC.set_ylabel("Number of pairs")
axC.tick_params(axis="x", rotation=20)
axC.grid(axis="y", alpha=0.25)
axC.grid(axis="x", visible=False)

for i, r in class_counts.iterrows():
    axC.text(
        i,
        r["count"] + class_counts["count"].max() * 0.02,
        f"{int(r['count']):,}",
        ha="center",
        fontsize=9,
        fontweight="bold",
    )

# -------------------------
# Panel D: QC summary table
# -------------------------
axD.axis("off")

qc_table = pd.DataFrame([
    {"QC item": "Final labeled pairs", "Value": f"{len(pairs_all):,}"},
    {"QC item": "Unique pair IDs", "Value": f"{pairs_all['pair_id'].nunique():,}"},
    {"QC item": "Positive labels", "Value": f"{int((pairs_all['label'].astype(int) == 1).sum()):,}"},
    {"QC item": "Negative labels", "Value": f"{int((pairs_all['label'].astype(int) == 0).sum()):,}"},
    {"QC item": "Duplicate pair IDs", "Value": f"{len(pairs_all) - pairs_all['pair_id'].nunique():,}"},
    {"QC item": "Unique enzymes", "Value": f"{pairs_all['enz_ac'].nunique():,}"},
    {"QC item": "Unique substrates", "Value": f"{pairs_all['sub_ac'].nunique():,}"},
])

tbl = axD.table(
    cellText=qc_table.values,
    colLabels=qc_table.columns,
    cellLoc="center",
    colLoc="center",
    loc="center",
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.15, 1.8)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor("#2F5597")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F7F7F7" if row % 2 == 0 else "white")

axD.set_title(
    "D. Final dataset QC summary",
    loc="left",
    fontsize=14,
    fontweight="bold",
    pad=20,
)

fig.suptitle(
    "Figure 3. Construction and quality control of the E3/DUB interaction dataset",
    fontsize=18,
    fontweight="bold",
    y=0.985,
)

plt.subplots_adjust(top=0.92, bottom=0.04, left=0.06, right=0.98)

save_final_figure(
    fig,
    3,
    "Figure3_dataset_construction_QC"
)

plt.show()

بریم Figure 4: Classical ML + Cross-model Fusion Benchmark.

این Figure نشان می‌دهد قبل از NN و GNN، مدل‌های کلاسیک و fusion چقدر خوب شدند.

Cell 10 — آماده‌سازی داده Figure 4

In [ ]:
# Figure 4 — Classical ML and Cross-model fusion benchmark, clean labels

baseline = data["baseline"].copy()
cross_model = data["cross_model"].copy()

label_map = {
    "esm2_t6_8M_UR50D": "ESM2-8M",
    "esm2_t12_35M_UR50D": "ESM2-35M",
    "esm2_t30_150M_UR50D": "ESM2-150M",
    "esm2_t33_650M_UR50D": "ESM2-650M",
    "esm2_t36_3B_UR50D": "ESM2-3B",
    "prot_bert": "ProtBERT",
    "prot_bert_bfd": "ProtBERT-BFD",
}

baseline["model_label"] = baseline["embedding_model"].map(label_map)

best_by_algorithm = (
    baseline
    .sort_values("pr_auc_mean", ascending=False)
    .groupby("algorithm", as_index=False)
    .first()
    .sort_values("pr_auc_mean", ascending=False)
)

best_pair_models = (
    baseline
    .sort_values("pr_auc_mean", ascending=False)
    .head(8)
    .copy()
)

best_pair_models["full_label"] = (
    best_pair_models["model_label"].astype(str)
    + " | "
    + best_pair_models["feature_type"].astype(str)
    + " | "
    + best_pair_models["algorithm"].astype(str)
)

cross_model["full_label"] = (
    cross_model["fusion_name"]
    .astype(str)
    .str.replace("__", " + ", regex=False)
    .str.replace("esm650", "ESM650", regex=False)
    .str.replace("esm3b", "ESM3B", regex=False)
    .str.replace("protbert_bfd", "ProtBERT-BFD", regex=False)
    + " | "
    + cross_model["algorithm"].astype(str)
)

best_fusion = (
    cross_model
    .sort_values("pr_auc_mean", ascending=False)
    .head(6)
    .copy()
)

best_pair_models = best_pair_models.reset_index(drop=True)
best_pair_models["code"] = [f"P{i+1}" for i in range(len(best_pair_models))]

best_fusion = best_fusion.reset_index(drop=True)
best_fusion["code"] = [f"F{i+1}" for i in range(len(best_fusion))]

pair_mapping = best_pair_models[
    ["code", "full_label", "roc_auc_mean", "pr_auc_mean", "brier_mean"]
].copy()

fusion_mapping = best_fusion[
    ["code", "full_label", "roc_auc_mean", "pr_auc_mean", "brier_mean"]
].copy()

pair_mapping["group"] = "Pair-level ML"
fusion_mapping["group"] = "Cross-model fusion"

figure4_mapping = pd.concat([pair_mapping, fusion_mapping], ignore_index=True)

figure4_summary = pd.concat([
    best_pair_models.assign(group="Pair-level ML")[
        ["group", "code", "full_label", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "pr_auc_std",
         "f1_mean", "accuracy_mean", "precision_mean", "recall_mean", "brier_mean"]
    ],
    best_fusion.assign(group="Cross-model fusion")[
        ["group", "code", "full_label", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "pr_auc_std",
         "f1_mean", "accuracy_mean", "precision_mean", "recall_mean", "brier_mean"]
    ],
], ignore_index=True)

figure4_summary = figure4_summary.sort_values("pr_auc_mean", ascending=False)

figure4_summary.to_csv(
    DAY19_DIR / "Figure_4" / "figure4_classical_ml_fusion_summary_clean.csv",
    index=False
)

figure4_mapping.to_csv(
    DAY19_DIR / "Figure_4" / "figure4_model_code_mapping.csv",
    index=False
)

display(best_by_algorithm)
display(figure4_summary)
display(figure4_mapping)

Cell 11 — Figure 4A فقط Pair-level ML، تمیز و خوانا

In [ ]:
fig = plt.figure(figsize=(22, 16), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.0, 1.15, 1.05],
    width_ratios=[1, 1],
    hspace=0.48,
    wspace=0.32
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, :])
axD = fig.add_subplot(gs[2, :])

# -------------------------
# Panel A
# -------------------------
algo_plot = best_by_algorithm.sort_values("pr_auc_mean", ascending=True)

algo_colors = {
    "xgboost": "#C00000",
    "random_forest": "#2F5597",
    "logistic": "#70AD47",
}

axA.barh(
    algo_plot["algorithm"],
    algo_plot["pr_auc_mean"],
    xerr=algo_plot["pr_auc_std"],
    color=[algo_colors.get(a, "#7F7F7F") for a in algo_plot["algorithm"]],
    edgecolor="black",
    linewidth=0.9,
    capsize=4
)

axA.set_xlabel("Best PR-AUC")
axA.set_ylabel("")
axA.set_title("A. Best pair-level algorithm performance", loc="left", fontweight="bold")
axA.set_xlim(algo_plot["pr_auc_mean"].min() - 0.045, algo_plot["pr_auc_mean"].max() + 0.04)

for i, r in algo_plot.reset_index(drop=True).iterrows():
    axA.text(
        r["pr_auc_mean"] + 0.006,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

axA.grid(axis="x", alpha=0.25)
axA.grid(axis="y", visible=False)

# -------------------------
# Panel B
# -------------------------
pair_plot = best_pair_models.sort_values("pr_auc_mean", ascending=True)

axB.barh(
    pair_plot["code"],
    pair_plot["pr_auc_mean"],
    xerr=pair_plot["pr_auc_std"],
    color="#2F5597",
    edgecolor="black",
    linewidth=0.9,
    capsize=4
)

axB.set_xlabel("PR-AUC")
axB.set_ylabel("Pair-level model code")
axB.set_title("B. Top pair-level configurations", loc="left", fontweight="bold")
axB.set_xlim(pair_plot["pr_auc_mean"].min() - 0.03, pair_plot["pr_auc_mean"].max() + 0.04)

for i, r in pair_plot.reset_index(drop=True).iterrows():
    axB.text(
        r["pr_auc_mean"] + 0.005,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

axB.grid(axis="x", alpha=0.25)
axB.grid(axis="y", visible=False)

# -------------------------
# Panel C
# -------------------------
fusion_plot = best_fusion.sort_values("pr_auc_mean", ascending=True)

axC.barh(
    fusion_plot["code"],
    fusion_plot["pr_auc_mean"],
    xerr=fusion_plot["pr_auc_std"],
    color="#ED7D31",
    edgecolor="black",
    linewidth=0.9,
    capsize=4
)

axC.set_xlabel("PR-AUC")
axC.set_ylabel("Fusion model code")
axC.set_title("C. Cross-model fusion benchmark", loc="left", fontweight="bold")
axC.set_xlim(fusion_plot["pr_auc_mean"].min() - 0.03, fusion_plot["pr_auc_mean"].max() + 0.04)

for i, r in fusion_plot.reset_index(drop=True).iterrows():
    axC.text(
        r["pr_auc_mean"] + 0.005,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

axC.grid(axis="x", alpha=0.25)
axC.grid(axis="y", visible=False)

# -------------------------
# Panel D - fixed title/table spacing
# -------------------------
axD.axis("off")

table_show = figure4_mapping.copy()
table_show = table_show[
    ["code", "group", "full_label", "roc_auc_mean", "pr_auc_mean", "brier_mean"]
]

table_show.columns = [
    "Code",
    "Group",
    "Model configuration",
    "ROC-AUC",
    "PR-AUC",
    "Brier"
]

for c in ["ROC-AUC", "PR-AUC", "Brier"]:
    table_show[c] = table_show[c].map(lambda x: f"{x:.3f}")

# عنوان را داخل axD نگذار؛ بیرون از جدول قرار بده
axD.text(
    0.0,
    1.08,
    "D. Model code mapping and performance summary",
    transform=axD.transAxes,
    fontsize=14,
    fontweight="bold",
    ha="left",
    va="bottom"
)

tbl = axD.table(
    cellText=table_show.values,
    colLabels=table_show.columns,
    cellLoc="center",
    colLoc="center",
    bbox=[0.0, 0.0, 1.0, 0.92],  # جدول را پایین‌تر و کمی کوتاه‌تر می‌کند
    colWidths=[0.07, 0.16, 0.48, 0.09, 0.09, 0.09]
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(9.2)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor("#2F5597")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F7F7F7" if row % 2 == 0 else "white")

        if col == 2:
            cell.set_text_props(ha="left")

plt.show()

بریم Figure 5: Neural Pair Scorer Benchmark.

این شکل باید نشان بدهد مدل‌های NN با lossهای مختلف چه عملکردی داشتند و چرا Focal / Hybrid بهتر شدند.

Cell 12 — آماده‌سازی داده Figure 5

In [ ]:
# Figure 5 — Neural pair scorer benchmark

nn_compare = data["nn_compare"].copy()

nn_fold_files = {
    "BCE": DATA_ML / "neural_pair_scorer_results" / "nn_bce_fold_metrics.csv",
    "Weighted BCE": DATA_ML / "neural_pair_scorer_results" / "nn_weighted_bce_fold_metrics.csv",
    "Focal": DATA_ML / "neural_pair_scorer_results" / "nn_focal_fold_metrics.csv",
    "V2 Focal": DATA_ML / "neural_pair_scorer_results" / "nn_v2_focal_fold_metrics.csv",
    "Hybrid Focal": DATA_ML / "neural_pair_scorer_results" / "nn_hybrid_focal_ranking_fold_metrics.csv",
}

nn_history_files = {
    "BCE": DATA_ML / "neural_pair_scorer_results" / "nn_bce_training_history.csv",
    "Weighted BCE": DATA_ML / "neural_pair_scorer_results" / "nn_weighted_bce_training_history.csv",
    "Focal": DATA_ML / "neural_pair_scorer_results" / "nn_focal_training_history.csv",
    "V2 Focal": DATA_ML / "neural_pair_scorer_results" / "nn_v2_focal_training_history.csv",
    "Hybrid Focal": DATA_ML / "neural_pair_scorer_results" / "nn_hybrid_focal_ranking_training_history.csv",
}

fold_dfs = []

for name, path in nn_fold_files.items():
    if path.exists():
        tmp = pd.read_csv(path)
        tmp["model_clean"] = name
        fold_dfs.append(tmp)
        print(name, tmp.shape)
    else:
        print("missing:", name, path)

nn_folds = pd.concat(fold_dfs, ignore_index=True)

history_dfs = []

for name, path in nn_history_files.items():
    if path.exists():
        tmp = pd.read_csv(path)
        tmp["model_clean"] = name
        history_dfs.append(tmp)
        print("history", name, tmp.shape)
    else:
        print("missing history:", name, path)

nn_history = pd.concat(history_dfs, ignore_index=True) if len(history_dfs) else None

display(nn_compare)
display(nn_folds.head())
if nn_history is not None:
    display(nn_history.head())

nn_folds.to_csv(DAY19_DIR / "Figure_5" / "figure5_nn_fold_metrics.csv", index=False)
if nn_history is not None:
    nn_history.to_csv(DAY19_DIR / "Figure_5" / "figure5_nn_training_history.csv", index=False)

Cell 13 — بررسی ستون‌های NN

In [ ]:
print("nn_compare columns:")
print(nn_compare.columns.tolist())

print("\nnn_folds columns:")
print(nn_folds.columns.tolist())

if nn_history is not None:
    print("\nnn_history columns:")
    print(nn_history.columns.tolist())

Cell 14 — آماده‌سازی خلاصه NN

In [ ]:
# Figure 5 — Neural pair scorer clean summary

nn_summary = (
    nn_folds
    .groupby("model_clean", as_index=False)
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        brier_mean=("brier", "mean"),
        brier_std=("brier", "std"),
        best_epoch_median=("best_epoch", "median"),
    )
)

nn_order = ["BCE", "Weighted BCE", "Focal", "V2 Focal", "Hybrid Focal"]
nn_summary["model_clean"] = pd.Categorical(
    nn_summary["model_clean"],
    categories=nn_order,
    ordered=True
)
nn_summary = nn_summary.sort_values("model_clean")

history_mean = (
    nn_history
    .groupby(["model_clean", "epoch"], as_index=False)
    .agg(
        train_loss_mean=("train_loss", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
        roc_auc_mean=("roc_auc", "mean"),
        brier_mean=("brier", "mean"),
    )
)

history_mean["model_clean"] = pd.Categorical(
    history_mean["model_clean"],
    categories=nn_order,
    ordered=True
)
history_mean = history_mean.sort_values(["model_clean", "epoch"])

nn_summary.to_csv(
    DAY19_DIR / "Figure_5" / "figure5_nn_summary.csv",
    index=False
)

history_mean.to_csv(
    DAY19_DIR / "Figure_5" / "figure5_nn_training_history_mean.csv",
    index=False
)

display(nn_summary)
display(history_mean.head())

Cell 15 — Figure 5 نهایی

In [ ]:
# Figure 5 — Neural Pair Scorer Benchmark

fig = plt.figure(figsize=(20, 16), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.05, 1.05, 0.95],
    width_ratios=[1, 1],
    hspace=0.46,
    wspace=0.34
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axE = fig.add_subplot(gs[2, :])

colors = {
    "BCE": "#2F5597",
    "Weighted BCE": "#70AD47",
    "Focal": "#C00000",
    "V2 Focal": "#ED7D31",
    "Hybrid Focal": "#7030A0",
}

# -------------------------
# Panel A: PR-AUC with fold std
# -------------------------
plotA = nn_summary.sort_values("pr_auc_mean", ascending=True)

axA.barh(
    plotA["model_clean"].astype(str),
    plotA["pr_auc_mean"],
    xerr=plotA["pr_auc_std"],
    color=[colors[x] for x in plotA["model_clean"].astype(str)],
    edgecolor="black",
    linewidth=0.8,
    capsize=4
)

axA.set_xlabel("PR-AUC")
axA.set_ylabel("")
axA.set_title("A. Neural loss comparison by PR-AUC", loc="left", fontweight="bold")
axA.set_xlim(plotA["pr_auc_mean"].min() - 0.025, plotA["pr_auc_mean"].max() + 0.025)

for i, r in plotA.reset_index(drop=True).iterrows():
    axA.text(
        r["pr_auc_mean"] + 0.003,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

axA.grid(axis="x", alpha=0.25)
axA.grid(axis="y", visible=False)

# -------------------------
# Panel B: metric profile
# -------------------------
metric_long = nn_summary.melt(
    id_vars="model_clean",
    value_vars=["roc_auc_mean", "pr_auc_mean", "f1_mean", "accuracy_mean"],
    var_name="metric",
    value_name="score"
)

metric_label = {
    "roc_auc_mean": "ROC-AUC",
    "pr_auc_mean": "PR-AUC",
    "f1_mean": "F1",
    "accuracy_mean": "Accuracy",
}
metric_long["metric"] = metric_long["metric"].map(metric_label)

sns.barplot(
    data=metric_long,
    x="metric",
    y="score",
    hue="model_clean",
    palette=colors,
    edgecolor="black",
    linewidth=0.5,
    ax=axB
)

axB.set_ylim(0.62, 0.88)
axB.set_xlabel("")
axB.set_ylabel("Score")
axB.set_title("B. Multi-metric neural benchmark", loc="left", fontweight="bold")
axB.legend(title="Loss", frameon=True, ncol=1, loc="lower right")
axB.grid(axis="y", alpha=0.25)
axB.grid(axis="x", visible=False)

# -------------------------
# Panel C: PR-AUC training trajectory
# -------------------------
for model in nn_order:
    tmp = history_mean[history_mean["model_clean"].astype(str) == model]
    if len(tmp) == 0:
        continue

    axC.plot(
        tmp["epoch"],
        tmp["pr_auc_mean"],
        label=model,
        color=colors[model],
        linewidth=2.0
    )

axC.set_xlabel("Epoch")
axC.set_ylabel("Validation PR-AUC")
axC.set_title("C. Mean validation PR-AUC trajectory", loc="left", fontweight="bold")
axC.legend(frameon=True, fontsize=8)
axC.grid(alpha=0.25)

# -------------------------
# Panel D: Brier score
# -------------------------
plotD = nn_summary.sort_values("brier_mean", ascending=False)

axD.barh(
    plotD["model_clean"].astype(str),
    plotD["brier_mean"],
    xerr=plotD["brier_std"],
    color=[colors[x] for x in plotD["model_clean"].astype(str)],
    edgecolor="black",
    linewidth=0.8,
    capsize=4
)

axD.set_xlabel("Brier score lower is better")
axD.set_ylabel("")
axD.set_title("D. Calibration-related error", loc="left", fontweight="bold")

for i, r in plotD.reset_index(drop=True).iterrows():
    axD.text(
        r["brier_mean"] + 0.003,
        i,
        f"{r['brier_mean']:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

axD.grid(axis="x", alpha=0.25)
axD.grid(axis="y", visible=False)

# -------------------------
# Panel E: table
# -------------------------
axE.axis("off")

axE.text(
    0.0,
    1.08,
    "E. Neural model performance summary",
    transform=axE.transAxes,
    fontsize=14,
    fontweight="bold",
    ha="left",
    va="bottom"
)

table_show = nn_summary[
    [
        "model_clean",
        "roc_auc_mean",
        "pr_auc_mean",
        "f1_mean",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "brier_mean",
        "best_epoch_median",
    ]
].copy()

table_show.columns = [
    "Model",
    "ROC-AUC",
    "PR-AUC",
    "F1",
    "Accuracy",
    "Precision",
    "Recall",
    "Brier",
    "Median best epoch",
]

for c in ["ROC-AUC", "PR-AUC", "F1", "Accuracy", "Precision", "Recall", "Brier"]:
    table_show[c] = table_show[c].map(lambda x: f"{x:.3f}")

table_show["Median best epoch"] = table_show["Median best epoch"].map(lambda x: f"{int(x)}")

tbl = axE.table(
    cellText=table_show.values,
    colLabels=table_show.columns,
    cellLoc="center",
    colLoc="center",
    bbox=[0.0, 0.0, 1.0, 0.92],
    colWidths=[0.16, 0.095, 0.095, 0.085, 0.095, 0.095, 0.095, 0.085, 0.13]
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(9.4)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor("#2F5597")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F7F7F7" if row % 2 == 0 else "white")

fig.suptitle(
    "Figure 5. Neural pair scorer benchmark across loss functions",
    fontsize=18,
    fontweight="bold",
    y=0.985,
)

plt.subplots_adjust(top=0.92, bottom=0.04, left=0.07, right=0.98)

save_final_figure(
    fig,
    5,
    "Figure5_neural_pair_scorer_benchmark"
)

plt.show()

بریم Figure 6: Graph Learning Benchmark.

این شکل باید نقطه اوج مدل‌سازی را نشان بدهد: از GraphSAGE و GAT تا HGT و MegaWeighted Ensemble.

Cell 16 — آماده‌سازی داده Figure 6

In [ ]:
# Figure 6 — Graph learning benchmark

graph_leaderboard = data["graph_leaderboard"].copy()
mega = data["mega"].copy()

def clean_graph_label(name):
    name = str(name)

    mapping = {
        "GraphSAGE": "GraphSAGE",
        "GAT": "GAT",
        "HeteroGraphSAGE": "HeteroGraphSAGE",
        "HeteroGAT": "HeteroGAT",
        "EdgeAware": "Edge-aware HeteroGNN",
        "edge_aware": "Edge-aware HeteroGNN",
        "HGT": "HGT",
        "hgt_only": "HGT",
        "hgt0.5_edge0.3_hetero0.2": "MegaWeighted Ensemble",
    }

    lower = name.lower()

    if name in mapping:
        return mapping[name]

    if "hgt0.5_edge0.3_hetero0.2" in lower:
        return "MegaWeighted Ensemble"
    if "hgt_only" in lower:
        return "HGT"
    if "hgt" in lower and "edge" in lower and "hetero" in lower:
        return "MegaWeighted Ensemble"
    if "edge" in lower:
        return "Edge-aware HeteroGNN"
    if "hetero_graphsage" in lower or "heterographsage" in lower:
        return "HeteroGraphSAGE"
    if "hetero_gat" in lower or "heterogat" in lower:
        return "HeteroGAT"
    if "graphsage" in lower:
        return "GraphSAGE"
    if "gat" in lower:
        return "GAT"

    return name


graph_core = graph_leaderboard.copy()
graph_core["source"] = "Graph models"
graph_core["clean_model"] = graph_core["model"].apply(clean_graph_label)

mega_core = mega.copy()
mega_core["source"] = "Final ensemble"
mega_core["clean_model"] = mega_core["model"].apply(clean_graph_label)

graph_all = pd.concat(
    [
        graph_core[
            [
                "source", "clean_model", "model",
                "roc_auc_mean", "roc_auc_std",
                "pr_auc_mean", "pr_auc_std",
                "f1_mean", "f1_std",
                "accuracy_mean", "accuracy_std",
                "precision_mean", "precision_std",
                "recall_mean", "recall_std",
                "brier_mean", "brier_std",
            ]
        ],
        mega_core[
            [
                "source", "clean_model", "model",
                "roc_auc_mean", "roc_auc_std",
                "pr_auc_mean", "pr_auc_std",
                "f1_mean", "f1_std",
                "accuracy_mean", "accuracy_std",
                "precision_mean", "precision_std",
                "recall_mean", "recall_std",
                "brier_mean", "brier_std",
            ]
        ],
    ],
    ignore_index=True
)

# بهترین ردیف برای هر مدل تمیز
graph_best = (
    graph_all
    .sort_values("pr_auc_mean", ascending=False)
    .groupby("clean_model", as_index=False)
    .first()
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

# حذف مدل‌های weighted تکراری که clean_label خاص ندارند
keep_models = [
    "MegaWeighted Ensemble",
    "HGT",
    "Edge-aware HeteroGNN",
    "HeteroGraphSAGE",
    "HeteroGAT",
    "GraphSAGE",
    "GAT",
]

graph_best = graph_best[
    graph_best["clean_model"].isin(keep_models)
].copy()

graph_best["clean_model"] = pd.Categorical(
    graph_best["clean_model"],
    categories=keep_models[::-1],
    ordered=True
)

graph_best = graph_best.sort_values("clean_model")

graph_best.to_csv(
    DAY19_DIR / "Figure_6" / "figure6_graph_model_benchmark_summary.csv",
    index=False
)

display(graph_best)

In [ ]:
graph_best.loc[
    graph_best["clean_model"]=="GAT",
    [
        "roc_auc_mean",
        "pr_auc_mean",
        "f1_mean",
        "accuracy_mean",
        "precision_mean",
        "recall_mean"
    ]
]

Cell 17 — Figure 6 نهایی

In [ ]:
# Figure 6 — Graph Learning Benchmark, final version with radar panel

fig = plt.figure(figsize=(22, 17), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.05, 1.05, 0.95],
    width_ratios=[1, 1.05],
    hspace=0.48,
    wspace=0.34
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0:2, 1], polar=True)
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[2, 0])
axE = fig.add_subplot(gs[2, 1])

graph_colors = {
    "GraphSAGE": "#2F5597",
    "GAT": "#70AD47",
    "HeteroGraphSAGE": "#ED7D31",
    "HeteroGAT": "#A5A5A5",
    "Edge-aware HeteroGNN": "#7030A0",
    "HGT": "#C00000",
    "MegaWeighted Ensemble": "#1F1F1F",
}

# -------------------------
# Panel A: PR-AUC ranking
# -------------------------
plotA = graph_best.sort_values("pr_auc_mean", ascending=True)

axA.barh(
    plotA["clean_model"].astype(str),
    plotA["pr_auc_mean"],
    xerr=plotA["pr_auc_std"],
    color=[graph_colors.get(x, "#7F7F7F") for x in plotA["clean_model"].astype(str)],
    edgecolor="black",
    linewidth=0.8,
    capsize=4
)

axA.set_xlabel("PR-AUC")
axA.set_ylabel("")
axA.set_title("A. Graph model ranking by PR-AUC", loc="left", fontweight="bold")
axA.set_xlim(plotA["pr_auc_mean"].min() - 0.04, plotA["pr_auc_mean"].max() + 0.025)

for i, r in plotA.reset_index(drop=True).iterrows():
    axA.text(
        r["pr_auc_mean"] + 0.004,
        i,
        f"{r['pr_auc_mean']:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

axA.grid(axis="x", alpha=0.25)
axA.grid(axis="y", visible=False)

# -------------------------
# Panel B: radar chart
# -------------------------
radar_df = graph_best.copy()
radar_df["calibration"] = 1 - radar_df["brier_mean"]

radar_models = [
    "MegaWeighted Ensemble",
    "HGT",
    "Edge-aware HeteroGNN",
    "HeteroGraphSAGE",
    "GraphSAGE",
    "GAT",
]

radar_df = radar_df[
    radar_df["clean_model"].astype(str).isin(radar_models)
].copy()

radar_df["clean_model"] = pd.Categorical(
    radar_df["clean_model"].astype(str),
    categories=radar_models,
    ordered=True
)

radar_df = radar_df.sort_values("clean_model")

radar_metrics = [
    "roc_auc_mean",
    "pr_auc_mean",
    "f1_mean",
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
    "calibration",
]

radar_labels = [
    "ROC",
    "PR",
    "F1",
    "ACC",
    "PREC",
    "REC",
    "CAL"
]

angles = np.linspace(
    0,
    2 * np.pi,
    len(radar_labels),
    endpoint=False
).tolist()

angles += angles[:1]

for _, row in radar_df.iterrows():
    model = str(row["clean_model"])

    values = [float(row[m]) for m in radar_metrics]
    values += values[:1]

    linewidth = 3.0 if model == "MegaWeighted Ensemble" else 1.8
    alpha = 0.18 if model == "MegaWeighted Ensemble" else 0.08

    axB.plot(
        angles,
        values,
        linewidth=linewidth,
        color=graph_colors.get(model, "#7F7F7F"),
        label=model
    )

    axB.fill(
        angles,
        values,
        color=graph_colors.get(model, "#7F7F7F"),
        alpha=alpha
    )

axB.set_xticks(angles[:-1])
axB.set_xticklabels(
    radar_labels,
    fontsize=11,
    fontweight="bold"
)

axB.set_ylim(0.70, 1.00)
axB.set_yticks([0.70, 0.80, 0.90, 1.00])
axB.set_yticklabels(["0.70", "0.80", "0.90", "1.00"], fontsize=8)
axB.grid(alpha=0.25)

axB.set_title(
    "B. Multi-metric graph model profile",
    loc="left",
    fontsize=13,
    fontweight="bold",
    pad=28
)

axB.legend(
    loc="upper left",
    bbox_to_anchor=(1.08, 1.08),
    frameon=False,
    fontsize=9
)

# -------------------------
# Panel C: multi-metric line profile - fixed y-axis
# -------------------------
metric_long = graph_best.melt(
    id_vars="clean_model",
    value_vars=[
        "roc_auc_mean",
        "pr_auc_mean",
        "f1_mean",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
    ],
    var_name="metric",
    value_name="score"
)

metric_map = {
    "roc_auc_mean": "ROC-AUC",
    "pr_auc_mean": "PR-AUC",
    "f1_mean": "F1",
    "accuracy_mean": "Accuracy",
    "precision_mean": "Precision",
    "recall_mean": "Recall",
}

metric_order = [
    "ROC-AUC",
    "PR-AUC",
    "F1",
    "Accuracy",
    "Precision",
    "Recall",
]

metric_long["metric"] = metric_long["metric"].map(metric_map)

metric_long["metric"] = pd.Categorical(
    metric_long["metric"],
    categories=metric_order,
    ordered=True
)

metric_long = metric_long.sort_values(["clean_model", "metric"])

sns.lineplot(
    data=metric_long,
    x="metric",
    y="score",
    hue="clean_model",
    marker="o",
    linewidth=2.0,
    markersize=7,
    palette=graph_colors,
    ax=axC,
    legend=False
)

# y-axis را امن و پویا می‌کنیم
y_min = metric_long["score"].min()
y_max = metric_long["score"].max()

axC.set_ylim(
    max(0.0, y_min - 0.08),
    min(1.0, y_max + 0.04)
)

axC.set_xlabel("")
axC.set_ylabel("Score")
axC.set_title(
    "C. Metric profile across graph models",
    loc="left",
    fontweight="bold"
)

axC.tick_params(axis="x", rotation=25)
axC.grid(axis="y", alpha=0.25)
axC.grid(axis="x", visible=False)

# -------------------------
# Panel D: Brier score
# -------------------------
plotD = graph_best.sort_values("brier_mean", ascending=False)

axD.barh(
    plotD["clean_model"].astype(str),
    plotD["brier_mean"],
    xerr=plotD["brier_std"],
    color=[graph_colors.get(x, "#7F7F7F") for x in plotD["clean_model"].astype(str)],
    edgecolor="black",
    linewidth=0.8,
    capsize=4
)

axD.set_xlabel("Brier score lower is better")
axD.set_ylabel("")
axD.set_title("D. Calibration-related error", loc="left", fontweight="bold")

for i, r in plotD.reset_index(drop=True).iterrows():
    axD.text(
        r["brier_mean"] + 0.003,
        i,
        f"{r['brier_mean']:.3f}",
        va="center",
        fontsize=9,
        fontweight="bold"
    )

axD.grid(axis="x", alpha=0.25)
axD.grid(axis="y", visible=False)

# -------------------------
# Panel E: compact table
# -------------------------
axE.axis("off")

axE.text(
    0.0,
    1.08,
    "E. Graph model performance summary",
    transform=axE.transAxes,
    fontsize=13,
    fontweight="bold",
    ha="left",
    va="bottom"
)

table_show = graph_best[
    [
        "clean_model",
        "roc_auc_mean",
        "pr_auc_mean",
        "f1_mean",
        "accuracy_mean",
        "brier_mean",
    ]
].copy()

table_show.columns = [
    "Model",
    "ROC",
    "PR",
    "F1",
    "ACC",
    "Brier",
]

for c in ["ROC", "PR", "F1", "ACC", "Brier"]:
    table_show[c] = table_show[c].map(lambda x: f"{x:.3f}")

tbl = axE.table(
    cellText=table_show.values,
    colLabels=table_show.columns,
    cellLoc="center",
    colLoc="center",
    bbox=[0.0, 0.0, 1.0, 0.92],
    colWidths=[0.36, 0.12, 0.12, 0.11, 0.11, 0.12]
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(8.8)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor("#2F5597")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F7F7F7" if row % 2 == 0 else "white")

fig.suptitle(
    "Figure 6. Graph neural network and ensemble benchmark",
    fontsize=18,
    fontweight="bold",
    y=0.985,
)

plt.subplots_adjust(top=0.92, bottom=0.04, left=0.07, right=0.96)

save_final_figure(
    fig,
    6,
    "Figure6_graph_learning_ensemble_benchmark_radar"
)

plt.show()

بریم Figure 7: Explainability.

این Figure باید نشان بدهد مدل فقط خوب پیش‌بینی نکرده، بلکه می‌توانیم بفهمیم از چه رابطه‌ها و چه نودهایی برای تصمیم‌گیری استفاده کرده.

Cell 18 — آماده‌سازی داده Figure 7

In [ ]:
# Figure 7 — Explainability

edge_importance = pd.read_csv(
    DAY13_DIR / "edge_type_importance_from_ablation.csv"
)

relation_importance = pd.read_csv(
    DAY17_DIR / "novel_candidate_relation_importance_percent.csv"
)

case_node_importance = pd.read_csv(
    DAY13_DIR / "case_study_node_importance.csv"
)

top_enrichment = pd.read_csv(
    DAY13_DIR / "top_prediction_enrichment.csv"
)

print("edge_importance:", edge_importance.shape)
print(edge_importance.columns.tolist())
display(edge_importance.head())

print("relation_importance:", relation_importance.shape)
print(relation_importance.columns.tolist())
display(relation_importance.head())

print("case_node_importance:", case_node_importance.shape)
print(case_node_importance.columns.tolist())
display(case_node_importance.head())

print("top_enrichment:", top_enrichment.shape)
print(top_enrichment.columns.tolist())
display(top_enrichment.head())

Cell 19 — آماده‌سازی داده Figure 7

In [ ]:
# Figure 7 — Explainability data preparation

edge_plot = edge_importance.copy()
edge_plot = edge_plot.sort_values("relative_importance_percent", ascending=True)

relation_plot = relation_importance.copy()
relation_plot["candidate"] = (
    relation_plot["enzyme"].astype(str)
    + "→"
    + relation_plot["substrate"].astype(str)
)

relation_stack = relation_plot[
    [
        "candidate",
        "ppi_importance_pct",
        "colocalization_importance_pct",
        "enzyme_substrate_context_importance_pct"
    ]
].copy()

relation_stack = relation_stack.rename(columns={
    "ppi_importance_pct": "PPI",
    "colocalization_importance_pct": "Co-localization",
    "enzyme_substrate_context_importance_pct": "Enzyme-substrate context"
})

case_pair = "DUB|Q14694|P04637"

case_nodes = (
    case_node_importance[
        case_node_importance["pair_id"] == case_pair
    ]
    .sort_values("importance", ascending=False)
    .head(15)
    .copy()
)

top_enrichment_plot = top_enrichment.copy()
top_enrichment_plot["k"] = (
    top_enrichment_plot["metric"]
    .astype(str)
    .str.replace("Top", "", regex=False)
    .astype(int)
)

explainability_summary = pd.DataFrame([
    {
        "Item": "Most important relation",
        "Result": edge_importance.sort_values("relative_importance_percent", ascending=False).iloc[0]["relation"]
    },
    {
        "Item": "PPI contribution",
        "Result": f"{edge_importance.loc[edge_importance['relation']=='PPI', 'relative_importance_percent'].iloc[0]:.1f}%"
    },
    {
        "Item": "Case study",
        "Result": "USP10→TP53"
    },
    {
        "Item": "Top node in case study",
        "Result": case_nodes.iloc[0]["gene"]
    },
    {
        "Item": "Top50 known interaction rate",
        "Result": f"{top_enrichment.loc[top_enrichment['metric']=='Top50', 'known_interaction_rate'].iloc[0]:.1f}%"
    },
])

edge_plot.to_csv(DAY19_DIR / "Figure_7" / "figure7_edge_importance.csv", index=False)
relation_stack.to_csv(DAY19_DIR / "Figure_7" / "figure7_relation_dependency_candidates.csv", index=False)
case_nodes.to_csv(DAY19_DIR / "Figure_7" / "figure7_case_node_importance_USP10_TP53.csv", index=False)
explainability_summary.to_csv(DAY19_DIR / "Figure_7" / "figure7_explainability_summary.csv", index=False)

display(edge_plot)
display(relation_stack)
display(case_nodes)
display(explainability_summary)

Cell 20 — Figure 7 نهایی

In [ ]:
# Figure 7 — Explainability final figure

fig = plt.figure(figsize=(22, 17), constrained_layout=False)

gs = GridSpec(
    4, 2,
    figure=fig,
    height_ratios=[1.0, 1.45, 1.15, 0.90],
    width_ratios=[1.0, 1.15],
    hspace=0.45,
    wspace=0.38
)

axA = fig.add_subplot(gs[0,0])

# بزرگ‌تر از قبل
axB = fig.add_subplot(gs[0:2,1])

axC = fig.add_subplot(gs[1,0])

axD = fig.add_subplot(gs[2,0])

axE = fig.add_subplot(gs[2:,1])

relation_colors = {
    "PPI": "#2F5597",
    "Co-localization": "#70AD47",
    "Enzyme-Substrate Context": "#C00000",
    "Enzyme-substrate context": "#C00000",
}

# -------------------------
# Panel A: global edge importance
# -------------------------
axA.barh(
    edge_plot["relation"],
    edge_plot["relative_importance_percent"],
    color=[
        relation_colors.get(r, "#7F7F7F")
        for r in edge_plot["relation"]
    ],
    edgecolor="black",
    linewidth=0.8
)

axA.set_xlabel("Relative importance (%)")
axA.set_ylabel("")
axA.set_title(
    "A. Global relation importance from edge ablation",
    loc="left",
    fontweight="bold"
)

for i, r in edge_plot.reset_index(drop=True).iterrows():
    axA.text(
        r["relative_importance_percent"] + 1.0,
        i,
        f"{r['relative_importance_percent']:.1f}%",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

axA.set_xlim(0, max(edge_plot["relative_importance_percent"]) + 12)
axA.grid(axis="x", alpha=0.25)
axA.grid(axis="y", visible=False)

# --------------------------------------------------
# Panel B
# Relation dependency across selected novel candidates
# --------------------------------------------------

stack_df = relation_stack.copy()

stack_df = stack_df.sort_values(
    "PPI",
    ascending=True
)

y = np.arange(len(stack_df))

left = np.zeros(len(stack_df))

relations = [
    "PPI",
    "Co-localization",
    "Enzyme-substrate context"
]

for relation in relations:

    axB.barh(
        y,
        stack_df[relation],
        left=left,
        color=relation_colors[relation],
        edgecolor="black",
        linewidth=0.5,
        height=0.72,
        label=relation
    )

    left += stack_df[relation].values


axB.set_yticks(y)

axB.set_yticklabels(
    stack_df["candidate"],
    fontsize=9,
    fontweight="bold"
)

axB.set_xlim(0,100)

axB.set_xlabel(
    "Relative contribution (%)",
    fontsize=11
)

axB.set_title(
    "B. Relation dependency across selected novel candidates",
    loc="left",
    fontweight="bold",
    fontsize=13
)

axB.grid(
    axis="x",
    alpha=0.25
)

axB.grid(
    axis="y",
    visible=False
)

# ---------- Legend خارج از نمودار ----------

axB.legend(
    title="Relation type",
    fontsize=9,
    title_fontsize=10,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02,1.00),
    borderaxespad=0
)

# کمی فضای خالی سمت راست
axB.margins(y=0.08)

# -------------------------
# Panel C: top prediction known-rate
# -------------------------
axC.plot(
    top_enrichment_plot["k"],
    top_enrichment_plot["known_interaction_rate"],
    marker="o",
    linewidth=2.6,
    markersize=8,
    color="#2F5597"
)

for _, r in top_enrichment_plot.iterrows():
    axC.text(
        r["k"],
        r["known_interaction_rate"] + 0.35,
        f"{r['known_interaction_rate']:.1f}%",
        ha="center",
        fontsize=9,
        fontweight="bold"
    )

axC.set_xscale("log")
axC.set_xlabel("Top-K predictions")
axC.set_ylabel("Known interaction rate (%)")
axC.set_title(
    "C. Known interaction recovery among top-ranked predictions",
    loc="left",
    fontweight="bold"
)
axC.set_ylim(95, 101)
axC.grid(alpha=0.25)

# -------------------------
# Panel D: case node importance
# -------------------------
case_plot = case_nodes.sort_values("importance", ascending=True)

axD.barh(
    case_plot["gene"],
    case_plot["importance"],
    color="#7030A0",
    edgecolor="black",
    linewidth=0.7
)

axD.set_xlabel("Node importance")
axD.set_ylabel("")
axD.set_title(
    "D. Case-level node importance for USP10→TP53",
    loc="left",
    fontweight="bold"
)

for i, r in case_plot.reset_index(drop=True).iterrows():
    axD.text(
        r["importance"] + case_plot["importance"].max() * 0.015,
        i,
        f"{r['importance']:.2e}",
        va="center",
        fontsize=8
    )

axD.grid(axis="x", alpha=0.25)
axD.grid(axis="y", visible=False)

# -------------------------
# Panel E: summary table
# -------------------------
axE.axis("off")

axE.text(
    0.0,
    1.08,
    "E. Explainability summary",
    transform=axE.transAxes,
    fontsize=14,
    fontweight="bold",
    ha="left",
    va="bottom"
)

tbl = axE.table(
    cellText=explainability_summary.values,
    colLabels=explainability_summary.columns,
    cellLoc="center",
    colLoc="center",
    bbox=[0.0, 0.0, 1.0, 0.92],
    colWidths=[0.35, 0.65]
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10.5)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor("#2F5597")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F7F7F7" if row % 2 == 0 else "white")

fig.suptitle(
    "Figure 7. Explainability of HGT-based E3/DUB specificity predictions",
    fontsize=18,
    fontweight="bold",
    y=0.985
)

plt.subplots_adjust(top=0.92, bottom=0.04, left=0.08, right=0.98)

save_final_figure(
    fig,
    7,
    "Figure7_HGT_explainability"
)

plt.show()

بریم Figure 8: Novel Prediction Discovery.

این Figure باید نشان بدهد مدل از فضای کاندیدها چطور به Novel candidateهای اولویت‌دار رسیده.

Cell 21 — آماده‌سازی داده Figure 8

In [ ]:
# Figure 8 — Novel prediction discovery

novel_top500 = pd.read_csv(DAY16_DIR / "top500_novel_predictions_hgt.csv")
novel_qc = pd.read_csv(DAY16_DIR / "day16_novel_prediction_qc.csv")
day17_priority = pd.read_csv(DAY17_DIR / "day17_final_novel_candidate_priority_table.csv")
relation_importance = pd.read_csv(DAY17_DIR / "novel_candidate_relation_importance_percent.csv")

print("novel_top500:", novel_top500.shape)
print("novel_qc:", novel_qc.shape)
print("day17_priority:", day17_priority.shape)
print("relation_importance:", relation_importance.shape)

display(novel_top500.head())
display(novel_qc)
display(day17_priority.head())

In [ ]:
print(day17_priority.columns.tolist())
display(day17_priority.head())

Cell 22 — ساخت جدول‌های Figure 8

In [ ]:
# Figure 8 — final robust plotting tables

top_novel = novel_top500.head(30).copy()

top_novel["candidate"] = (
    top_novel["enz_gene"].astype(str)
    + "→"
    + top_novel["sub_gene"].astype(str)
)

top_novel_plot = (
    top_novel
    .head(15)
    .sort_values("prob_hgt_mean", ascending=True)
    .copy()
)

priority = day17_priority.copy()

if "candidate" not in priority.columns:
    if {"enzyme", "substrate"}.issubset(priority.columns):
        priority["candidate"] = (
            priority["enzyme"].astype(str)
            + "→"
            + priority["substrate"].astype(str)
        )
    elif {"enz_gene", "sub_gene"}.issubset(priority.columns):
        priority["candidate"] = (
            priority["enz_gene"].astype(str)
            + "→"
            + priority["sub_gene"].astype(str)
        )

if "final_priority_score" not in priority.columns:
    score_candidates = [
        "priority_score",
        "prob_hgt_mean_day16",
        "prob_hgt_mean",
        "probability",
        "hgt_probability",
    ]

    score_col = None
    for c in score_candidates:
        if c in priority.columns:
            score_col = c
            break

    if score_col is None:
        raise ValueError(
            "No usable priority score column found. Available columns are:\n"
            + str(priority.columns.tolist())
        )

    priority["final_priority_score"] = priority[score_col]

priority_plot = (
    priority
    .sort_values("final_priority_score", ascending=False)
    .head(8)
    .sort_values("final_priority_score", ascending=True)
    .copy()
)

class_dist = (
    novel_top500["enzyme_class"]
    .value_counts()
    .reset_index()
)

class_dist.columns = ["enzyme_class", "count"]
class_dist["percent"] = class_dist["count"] / class_dist["count"].sum() * 100

enzyme_rank = (
    novel_top500
    .groupby(["enzyme_class", "enz_gene"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(12)
)

enzyme_rank["enzyme_label"] = (
    enzyme_rank["enzyme_class"].astype(str)
    + " | "
    + enzyme_rank["enz_gene"].astype(str)
)

top_priority_candidate = (
    priority
    .sort_values("final_priority_score", ascending=False)
    .iloc[0]
)

summary_table = pd.DataFrame([
    {
        "Item": "Scored candidate space",
        "Value": ">3,787,000,000 pairs",
    },
    {
        "Item": "Top novel predictions considered",
        "Value": f"{len(novel_top500):,}",
    },
    {
        "Item": "Highest HGT probability, Top500",
        "Value": f"{novel_top500['prob_hgt_mean'].max():.4f}",
    },
    {
        "Item": "Dominant enzyme in Top500",
        "Value": f"{enzyme_rank.iloc[0]['enz_gene']} ({enzyme_rank.iloc[0]['enzyme_class']})",
    },
    {
        "Item": "Top explainability-informed priority candidate",
        "Value": f"{top_priority_candidate['candidate']} (score = {top_priority_candidate['final_priority_score']:.3f})",
    },
])

top_novel_plot.to_csv(DAY19_DIR / "Figure_8" / "figure8_top_novel_candidates_final.csv", index=False)
priority_plot.to_csv(DAY19_DIR / "Figure_8" / "figure8_priority_candidates_final.csv", index=False)
class_dist.to_csv(DAY19_DIR / "Figure_8" / "figure8_class_distribution_final.csv", index=False)
enzyme_rank.to_csv(DAY19_DIR / "Figure_8" / "figure8_enzyme_rank_final.csv", index=False)
summary_table.to_csv(DAY19_DIR / "Figure_8" / "figure8_summary_table_final.csv", index=False)

display(top_novel_plot)
display(priority_plot)
display(class_dist)
display(enzyme_rank)
display(summary_table)

Cell 23 — Figure 8 نهایی

In [ ]:
# Figure 8 — Novel Prediction Discovery, publication-style final version

fig = plt.figure(figsize=(24, 16), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.20, 1.20, 0.65],
    width_ratios=[1.0, 1.05],
    hspace=0.38,
    wspace=0.28
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axE = fig.add_subplot(gs[2, :])

blue = "#164A8B"
red = "#D73027"
green = "#7EAD50"
purple = "#7B3294"

# -------------------------
# Panel A - fixed label placement
# -------------------------
axA.barh(
    top_novel_plot["candidate"],
    top_novel_plot["prob_hgt_mean"],
    xerr=top_novel_plot["prob_hgt_std"],
    color=blue,
    edgecolor="black",
    linewidth=0.8,
    capsize=3
)

axA.set_xlabel("Mean HGT probability", fontsize=11)
axA.set_ylabel("")
axA.set_title(
    "A. Top-ranked novel candidate interactions",
    loc="left",
    fontweight="bold",
    fontsize=13
)

xmin = 0.990
xmax = 1.0012
axA.set_xlim(xmin, xmax)

label_x = xmax - 0.00018

for i, r in top_novel_plot.reset_index(drop=True).iterrows():
    axA.text(
        label_x,
        i,
        f"{r['prob_hgt_mean']:.4f}",
        va="center",
        ha="right",
        fontsize=9,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.85,
            pad=1.2
        )
    )

axA.grid(axis="x", alpha=0.25, linestyle="--")
axA.grid(axis="y", alpha=0.15, linestyle="--")
axA.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel B
# -------------------------
axB.axis("equal")

class_dist_sorted = class_dist.sort_values("enzyme_class", ascending=True).copy()

pie_colors = [
    red if x == "E3" else blue
    for x in class_dist_sorted["enzyme_class"]
]

wedges, texts, autotexts = axB.pie(
    class_dist_sorted["count"],
    startangle=90,
    colors=pie_colors,
    autopct=lambda p: f"{p:.1f}%\n({int(round(p * class_dist_sorted['count'].sum() / 100))})",
    wedgeprops={"edgecolor": "black", "linewidth": 0.8},
    textprops={"fontsize": 12, "fontweight": "bold", "color": "white"}
)

for t in autotexts:
    t.set_fontsize(12)
    t.set_fontweight("bold")
    t.set_color("white")

legend_labels = [
    f"{row['enzyme_class']}\n{int(row['count'])} ({row['percent']:.1f}%)"
    for _, row in class_dist_sorted.iterrows()
]

axB.legend(
    wedges,
    legend_labels,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=11
)

axB.text(
    0,
    -1.25,
    f"Total = {class_dist_sorted['count'].sum():,}",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold"
)

axB.set_title(
    "B. Enzyme-class composition of Top500 novel predictions",
    loc="left",
    fontweight="bold",
    fontsize=13
)

# -------------------------
# Panel C
# -------------------------
enzyme_rank_plot = enzyme_rank.sort_values("count", ascending=True)

axC.barh(
    enzyme_rank_plot["enzyme_label"],
    enzyme_rank_plot["count"],
    color=purple,
    edgecolor="black",
    linewidth=0.8
)

axC.set_xlabel("Count in Top500", fontsize=11)
axC.set_ylabel("")
axC.set_title(
    "C. Most recurrent enzymes in novel predictions",
    loc="left",
    fontweight="bold",
    fontsize=13
)

for i, r in enzyme_rank_plot.reset_index(drop=True).iterrows():
    axC.text(
        r["count"] + 1.2,
        i,
        str(int(r["count"])),
        va="center",
        fontsize=10,
        fontweight="bold"
    )

axC.set_xlim(0, enzyme_rank_plot["count"].max() + 12)
axC.grid(axis="x", alpha=0.25, linestyle="--")
axC.grid(axis="y", visible=False)
axC.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel D
# -------------------------
priority_plot_fixed = priority_plot.copy()
priority_plot_fixed = priority_plot_fixed.sort_values("final_priority_score", ascending=True)

axD.barh(
    priority_plot_fixed["candidate"],
    priority_plot_fixed["final_priority_score"],
    color=green,
    edgecolor="black",
    linewidth=0.8
)

axD.set_xlabel("Final priority score", fontsize=11)
axD.set_ylabel("")
axD.set_title(
    "D. Explainability-informed priority candidates",
    loc="left",
    fontweight="bold",
    fontsize=13
)

# فضای راست بیشتر برای عددها
xmax = priority_plot_fixed["final_priority_score"].max()
axD.set_xlim(0, xmax * 1.12)

for i, r in priority_plot_fixed.reset_index(drop=True).iterrows():
    axD.text(
        r["final_priority_score"] + xmax * 0.018,
        i,
        f"{r['final_priority_score']:.3f}",
        va="center",
        ha="left",
        fontsize=10,
        fontweight="bold",
        clip_on=False
    )

axD.grid(axis="x", alpha=0.25, linestyle="--")
axD.grid(axis="y", visible=False)
axD.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel E
# -------------------------
axE.axis("off")

axE.text(
    0.0,
    1.05,
    "E. Novel discovery summary",
    transform=axE.transAxes,
    fontsize=13,
    fontweight="bold",
    ha="left",
    va="bottom"
)

tbl = axE.table(
    cellText=summary_table.values,
    colLabels=summary_table.columns,
    cellLoc="center",
    colLoc="center",
    bbox=[0.0, 0.0, 1.0, 0.92],
    colWidths=[0.50, 0.50]
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10.5)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("#A6A6A6")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor(blue)
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F3F6FA" if row % 2 == 1 else "white")
        cell.set_text_props(weight="bold")

fig.suptitle(
    "Figure 8. Novel E3/DUB–substrate interaction discovery",
    fontsize=20,
    fontweight="bold",
    y=0.985
)

plt.subplots_adjust(
    top=0.92,
    bottom=0.04,
    left=0.07,
    right=0.94
)

save_final_figure(
    fig,
    8,
    "Figure8_novel_prediction_discovery_publication_style"
)

plt.show()

بریم Figure 9 — Cancer-specific / GI prioritization.

این شکل بر اساس خروجی‌های Day 18 ساخته می‌شود:

Cell 24 — آماده‌سازی داده‌های Figure 9

In [ ]:
DAY18_DIR = GRAPH_DIR / "day18_cancer_specific_analysis"

In [ ]:
# Figure 9 — Cancer-specific GI prioritization

day18_priority = pd.read_csv(
    DAY18_DIR / "day18_final_gi_cancer_priority_candidates.csv"
)

driver_summary = pd.read_csv(
    DAY18_DIR / "driver_enrichment_summary.csv"
)

pathway_freq = pd.read_csv(
    DAY18_DIR / "gi_driver_pathway_frequency.csv"
)

crc_rank = pd.read_csv(
    DAY18_DIR / "crc_enzyme_rank.csv"
)

lihc_rank = pd.read_csv(
    DAY18_DIR / "lihc_enzyme_rank.csv"
)

gi_rank = pd.read_csv(
    DAY18_DIR / "gi_enzyme_rank.csv"
)

crc_specific = pd.read_csv(
    DAY18_DIR / "crc_specific_top_interactions.csv"
)

lihc_specific = pd.read_csv(
    DAY18_DIR / "lihc_specific_top_interactions.csv"
)

shared_gi = pd.read_csv(
    DAY18_DIR / "shared_gi_top_interactions.csv"
)

print("day18_priority:", day18_priority.shape)
print("driver_summary:", driver_summary.shape)
print("pathway_freq:", pathway_freq.shape)
print("crc_rank:", crc_rank.shape)
print("lihc_rank:", lihc_rank.shape)
print("gi_rank:", gi_rank.shape)
print("crc_specific:", crc_specific.shape)
print("lihc_specific:", lihc_specific.shape)
print("shared_gi:", shared_gi.shape)

display(day18_priority.head())
display(driver_summary)
display(pathway_freq)

Cell 25 — ساخت جدول‌های لازم برای Figure 9

In [ ]:
# Figure 9 — plotting tables

gi_top = day18_priority.copy()

gi_top["candidate"] = (
    gi_top["enz_gene"].astype(str)
    + "→"
    + gi_top["sub_gene"].astype(str)
)

gi_top_plot = (
    gi_top
    .sort_values("final_priority_score", ascending=False)
    .head(15)
    .sort_values("final_priority_score", ascending=True)
    .copy()
)

crc_rank_plot = (
    crc_rank
    .sort_values("count", ascending=False)
    .head(10)
    .sort_values("count", ascending=True)
    .copy()
)

lihc_rank_plot = (
    lihc_rank
    .sort_values("count", ascending=False)
    .head(10)
    .sort_values("count", ascending=True)
    .copy()
)

pathway_plot = (
    pathway_freq
    .sort_values("count", ascending=False)
    .copy()
)

summary9 = pd.DataFrame([
    {
        "Item": "GI priority candidates",
        "Value": f"{len(day18_priority):,}",
    },
    {
        "Item": "Shared CRC/LIHC candidates",
        "Value": f"{len(shared_gi):,}",
    },
    {
        "Item": "CRC-specific candidates",
        "Value": f"{len(crc_specific):,}",
    },
    {
        "Item": "LIHC-specific candidates",
        "Value": f"{len(lihc_specific):,}",
    },
    {
        "Item": "Dominant GI enzyme",
        "Value": f"{gi_rank.sort_values('count', ascending=False).iloc[0]['enz_gene']}",
    },
    {
        "Item": "Dominant driver pathway",
        "Value": f"{pathway_freq.sort_values('count', ascending=False).iloc[0]['pathway']}",
    },
    {
        "Item": "Top GI priority pair",
        "Value": f"{gi_top.sort_values('final_priority_score', ascending=False).iloc[0]['candidate']}",
    },
])

gi_top_plot.to_csv(DAY19_DIR / "Figure_9" / "figure9_top_gi_priority_candidates.csv", index=False)
crc_rank_plot.to_csv(DAY19_DIR / "Figure_9" / "figure9_crc_enzyme_rank.csv", index=False)
lihc_rank_plot.to_csv(DAY19_DIR / "Figure_9" / "figure9_lihc_enzyme_rank.csv", index=False)
pathway_plot.to_csv(DAY19_DIR / "Figure_9" / "figure9_pathway_frequency.csv", index=False)
summary9.to_csv(DAY19_DIR / "Figure_9" / "figure9_summary_table.csv", index=False)

display(gi_top_plot)
display(crc_rank_plot)
display(lihc_rank_plot)
display(pathway_plot)
display(summary9)

Cell 26 — Figure 9 نهایی

In [ ]:
# Figure 9 — Cancer-specific GI prioritization

fig = plt.figure(figsize=(24, 17), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.20, 1.20, 0.70],
    width_ratios=[1.05, 1.0],
    hspace=0.40,
    wspace=0.30
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axE = fig.add_subplot(gs[2, :])

blue = "#164A8B"
red = "#D73027"
green = "#7EAD50"
purple = "#7B3294"
orange = "#ED7D31"

# -------------------------
# Panel A: Top GI candidates
# -------------------------
axA.barh(
    gi_top_plot["candidate"],
    gi_top_plot["final_priority_score"],
    color=green,
    edgecolor="black",
    linewidth=0.8
)

xmax = gi_top_plot["final_priority_score"].max()
axA.set_xlim(0, xmax * 1.14)

for i, r in gi_top_plot.reset_index(drop=True).iterrows():
    axA.text(
        r["final_priority_score"] + xmax * 0.018,
        i,
        f"{r['final_priority_score']:.3f}",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold",
        clip_on=False
    )

axA.set_xlabel("Final GI priority score", fontsize=11)
axA.set_ylabel("")
axA.set_title(
    "A. Top GI cancer-prioritized novel candidates",
    loc="left",
    fontweight="bold",
    fontsize=13
)
axA.grid(axis="x", alpha=0.25, linestyle="--")
axA.grid(axis="y", visible=False)
axA.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel B: CRC vs LIHC enzyme ranking
# -------------------------
y_crc = np.arange(len(crc_rank_plot))
y_lihc = np.arange(len(lihc_rank_plot))

axB.barh(
    y_crc + 0.18,
    crc_rank_plot["count"],
    height=0.34,
    color=blue,
    edgecolor="black",
    linewidth=0.7,
    label="CRC"
)

axB.barh(
    y_lihc - 0.18,
    lihc_rank_plot["count"],
    height=0.34,
    color=red,
    edgecolor="black",
    linewidth=0.7,
    label="LIHC"
)

combined_labels = crc_rank_plot["enz_gene"].astype(str).tolist()

# اگر لیبل‌های CRC و LIHC فرق داشتند، این حالت امن‌تر است
max_len = max(len(crc_rank_plot), len(lihc_rank_plot))
axB.set_yticks(np.arange(max_len))

labels = []
for i in range(max_len):
    left_label = crc_rank_plot["enz_gene"].astype(str).tolist()[i] if i < len(crc_rank_plot) else ""
    right_label = lihc_rank_plot["enz_gene"].astype(str).tolist()[i] if i < len(lihc_rank_plot) else ""
    labels.append(f"{left_label} / {right_label}")

axB.set_yticklabels(labels, fontsize=9)

axB.set_xlabel("Count in Top cancer-specific candidates", fontsize=11)
axB.set_title(
    "B. CRC versus LIHC recurrent enzyme ranking",
    loc="left",
    fontweight="bold",
    fontsize=13
)
axB.legend(frameon=False, loc="lower right")
axB.grid(axis="x", alpha=0.25, linestyle="--")
axB.grid(axis="y", visible=False)

# -------------------------
# Panel C: pathway frequency
# -------------------------
pathway_plot_sorted = pathway_plot.sort_values("count", ascending=True)

axC.barh(
    pathway_plot_sorted["pathway"],
    pathway_plot_sorted["count"],
    color=purple,
    edgecolor="black",
    linewidth=0.8
)

for i, r in pathway_plot_sorted.reset_index(drop=True).iterrows():
    axC.text(
        r["count"] + pathway_plot_sorted["count"].max() * 0.018,
        i,
        str(int(r["count"])),
        va="center",
        ha="left",
        fontsize=10,
        fontweight="bold"
    )

axC.set_xlim(0, pathway_plot_sorted["count"].max() * 1.16)
axC.set_xlabel("Number of driver-linked candidates", fontsize=11)
axC.set_ylabel("")
axC.set_title(
    "C. Driver pathway frequency among GI candidates",
    loc="left",
    fontweight="bold",
    fontsize=13
)
axC.grid(axis="x", alpha=0.25, linestyle="--")
axC.grid(axis="y", visible=False)
axC.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel D: driver enrichment summary
# -------------------------
driver_plot = driver_summary.copy()

driver_plot["label"] = driver_plot["set"].astype(str)

axD.bar(
    driver_plot["label"],
    driver_plot["driver_rate"],
    color=orange,
    edgecolor="black",
    linewidth=0.8
)

for i, r in driver_plot.reset_index(drop=True).iterrows():
    axD.text(
        i,
        r["driver_rate"] + 2,
        f"{r['driver_rate']:.1f}%\n(n={int(r['n_driver'])})",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

axD.set_ylim(0, max(driver_plot["driver_rate"]) + 14)
axD.set_ylabel("Driver-hit rate (%)", fontsize=11)
axD.set_xlabel("")
axD.set_title(
    "D. Driver enrichment in cancer-prioritized sets",
    loc="left",
    fontweight="bold",
    fontsize=13
)
axD.tick_params(axis="x", rotation=20, labelsize=9)
axD.grid(axis="y", alpha=0.25, linestyle="--")
axD.grid(axis="x", visible=False)

# -------------------------
# Panel E: summary table
# -------------------------
axE.axis("off")

axE.text(
    0.0,
    1.05,
    "E. Cancer-specific prioritization summary",
    transform=axE.transAxes,
    fontsize=13,
    fontweight="bold",
    ha="left",
    va="bottom"
)

tbl = axE.table(
    cellText=summary9.values,
    colLabels=summary9.columns,
    cellLoc="center",
    colLoc="center",
    bbox=[0.0, 0.0, 1.0, 0.92],
    colWidths=[0.50, 0.50]
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10.5)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("#A6A6A6")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor(blue)
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F3F6FA" if row % 2 == 1 else "white")
        cell.set_text_props(weight="bold")

fig.suptitle(
    "Figure 9. Cancer-specific prioritization of novel E3/DUB–substrate interactions",
    fontsize=20,
    fontweight="bold",
    y=0.985
)

plt.subplots_adjust(
    top=0.92,
    bottom=0.04,
    left=0.08,
    right=0.96
)

save_final_figure(
    fig,
    9,
    "Figure9_cancer_specific_GI_prioritization"
)

plt.show()

بریم Figure 10 — Case studies and biological interpretation.

این Figure باید چند کاندید/تعامل شاخص را به‌صورت موردی نشان بدهد؛ یعنی از حالت benchmark خارج شود و بگوید: «مدل چه چیزهای زیستی معناداری پیدا کرده؟»

Figure 10 پیشنهادی

Panel A: Top case-study candidates
Panel B: Probability comparison برای caseها
Panel C: Pathway annotation برای caseها
Panel D: Literature / biological rationale table
Panel E: Final case-study summary

برای شروع، این Cell را اجرا کن تا ستون‌های فایل‌های لازم را دقیق ببینیم:

In [ ]:
# Figure 10 — Case study input inspection

case_study = pd.read_csv(
    GRAPH_DIR / "day14_case_studies" / "case_study_article_table.csv"
)

day18_priority = pd.read_csv(
    DAY18_DIR / "day18_final_gi_cancer_priority_candidates.csv"
)

day17_priority = pd.read_csv(
    DAY17_DIR / "day17_final_novel_candidate_priority_table.csv"
)

print("case_study:", case_study.shape)
print(case_study.columns.tolist())
display(case_study.head(20))

print("day18_priority:", day18_priority.shape)
print(day18_priority.columns.tolist())
display(day18_priority.head(20))

print("day17_priority:", day17_priority.shape)
print(day17_priority.columns.tolist())
display(day17_priority.head(20))

Cell 27 — آماده‌سازی داده Figure 10

In [ ]:
# Figure 10 — Case studies and biological interpretation

case_study = pd.read_csv(
    GRAPH_DIR / "day14_case_studies" / "case_study_article_table.csv"
)

day18_priority = pd.read_csv(
    DAY18_DIR / "day18_final_gi_cancer_priority_candidates.csv"
)

day17_priority = pd.read_csv(
    DAY17_DIR / "day17_final_novel_candidate_priority_table.csv"
)

# -------------------------
# Canonical/validated case studies
# -------------------------
case_plot = case_study.copy()
case_plot = case_plot.sort_values("probability", ascending=True)

# -------------------------
# Novel explainability-prioritized cases
# -------------------------
novel_cases = day17_priority.copy()

novel_cases["candidate"] = (
    novel_cases["enzyme"].astype(str)
    + "→"
    + novel_cases["substrate"].astype(str)
)

novel_cases_plot = (
    novel_cases
    .sort_values("priority_score", ascending=False)
    .head(8)
    .sort_values("priority_score", ascending=True)
    .copy()
)

# -------------------------
# GI cancer biological cases
# -------------------------
gi_cases = day18_priority.copy()

gi_cases["candidate"] = (
    gi_cases["enz_gene"].astype(str)
    + "→"
    + gi_cases["sub_gene"].astype(str)
)

gi_cases_plot = (
    gi_cases
    .sort_values("final_priority_score", ascending=False)
    .head(10)
    .copy()
)

# -------------------------
# Pathway distribution among top GI cases
# -------------------------
pathway_case = (
    gi_cases_plot["driver_pathways"]
    .fillna("unannotated")
    .str.split(";")
    .explode()
    .str.strip()
    .value_counts()
    .reset_index()
)

pathway_case.columns = ["pathway", "count"]

# -------------------------
# Summary table
# -------------------------
summary10 = pd.DataFrame([
    {
        "Item": "Highest canonical case probability",
        "Value": f"{case_study['probability'].max():.4f}",
    },
    {
        "Item": "Top canonical case",
        "Value": case_study.sort_values("probability", ascending=False).iloc[0]["pair_label"],
    },
    {
        "Item": "Top novel explainability case",
        "Value": novel_cases.sort_values("priority_score", ascending=False).iloc[0]["candidate"],
    },
    {
        "Item": "Top GI biological candidate",
        "Value": gi_cases.sort_values("final_priority_score", ascending=False).iloc[0]["candidate"],
    },
    {
        "Item": "Dominant pathway in top GI cases",
        "Value": pathway_case.iloc[0]["pathway"],
    },
])

case_plot.to_csv(DAY19_DIR / "Figure_10" / "figure10_canonical_case_probabilities.csv", index=False)
novel_cases_plot.to_csv(DAY19_DIR / "Figure_10" / "figure10_novel_explainability_cases.csv", index=False)
gi_cases_plot.to_csv(DAY19_DIR / "Figure_10" / "figure10_gi_biological_cases.csv", index=False)
pathway_case.to_csv(DAY19_DIR / "Figure_10" / "figure10_case_pathway_distribution.csv", index=False)
summary10.to_csv(DAY19_DIR / "Figure_10" / "figure10_summary_table.csv", index=False)

display(case_plot)
display(novel_cases_plot)
display(gi_cases_plot)
display(pathway_case)
display(summary10)

Cell 28 — Figure 10 نهایی

In [ ]:
# Figure 10 — Case studies and biological interpretation

fig = plt.figure(figsize=(24, 17), constrained_layout=False)

gs = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.20, 1.20, 0.70],
    width_ratios=[1.0, 1.0],
    hspace=0.40,
    wspace=0.30
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axE = fig.add_subplot(gs[2, :])

blue = "#164A8B"
red = "#D73027"
green = "#7EAD50"
purple = "#7B3294"
orange = "#ED7D31"

# -------------------------
# Panel A: canonical case probabilities
# -------------------------
axA.barh(
    case_plot["pair_label"],
    case_plot["probability"],
    color=blue,
    edgecolor="black",
    linewidth=0.8
)

axA.set_xlim(0.9957, 0.9964)

for i, r in case_plot.reset_index(drop=True).iterrows():
    axA.text(
        0.99637,
        i,
        f"{r['probability']:.4f}",
        va="center",
        ha="right",
        fontsize=9,
        fontweight="bold",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.2)
    )

axA.set_xlabel("HGT probability", fontsize=11)
axA.set_ylabel("")
axA.set_title(
    "A. Canonical biological case-study probabilities",
    loc="left",
    fontweight="bold",
    fontsize=13
)
axA.grid(axis="x", alpha=0.25, linestyle="--")
axA.grid(axis="y", visible=False)
axA.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel B: novel explainability priority
# -------------------------
axB.barh(
    novel_cases_plot["candidate"],
    novel_cases_plot["priority_score"],
    color=green,
    edgecolor="black",
    linewidth=0.8
)

xmax = novel_cases_plot["priority_score"].max()
axB.set_xlim(0, xmax * 1.12)

for i, r in novel_cases_plot.reset_index(drop=True).iterrows():
    axB.text(
        r["priority_score"] + xmax * 0.018,
        i,
        f"{r['priority_score']:.3f}",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold",
        clip_on=False
    )

axB.set_xlabel("Explainability-informed priority score", fontsize=11)
axB.set_ylabel("")
axB.set_title(
    "B. Novel explainability-prioritized candidates",
    loc="left",
    fontweight="bold",
    fontsize=13
)
axB.grid(axis="x", alpha=0.25, linestyle="--")
axB.grid(axis="y", visible=False)
axB.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel C: GI biological cases
# -------------------------
gi_cases_plot_sorted = gi_cases_plot.sort_values("final_priority_score", ascending=True)

axC.barh(
    gi_cases_plot_sorted["candidate"],
    gi_cases_plot_sorted["final_priority_score"],
    color=purple,
    edgecolor="black",
    linewidth=0.8
)

xmax = gi_cases_plot_sorted["final_priority_score"].max()
axC.set_xlim(0, xmax * 1.15)

for i, r in gi_cases_plot_sorted.reset_index(drop=True).iterrows():
    axC.text(
        r["final_priority_score"] + xmax * 0.018,
        i,
        f"{r['final_priority_score']:.3f}",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold",
        clip_on=False
    )

axC.set_xlabel("GI cancer priority score", fontsize=11)
axC.set_ylabel("")
axC.set_title(
    "C. Top GI cancer biological candidate cases",
    loc="left",
    fontweight="bold",
    fontsize=13
)
axC.grid(axis="x", alpha=0.25, linestyle="--")
axC.grid(axis="y", visible=False)
axC.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel D: pathway distribution
# -------------------------
pathway_case_plot = pathway_case.sort_values("count", ascending=True)

axD.barh(
    pathway_case_plot["pathway"],
    pathway_case_plot["count"],
    color=orange,
    edgecolor="black",
    linewidth=0.8
)

xmax = pathway_case_plot["count"].max()
axD.set_xlim(0, xmax * 1.25)

for i, r in pathway_case_plot.reset_index(drop=True).iterrows():
    axD.text(
        r["count"] + xmax * 0.04,
        i,
        str(int(r["count"])),
        va="center",
        ha="left",
        fontsize=10,
        fontweight="bold"
    )

axD.set_xlabel("Count among top GI case candidates", fontsize=11)
axD.set_ylabel("")
axD.set_title(
    "D. Biological pathway annotation of top cases",
    loc="left",
    fontweight="bold",
    fontsize=13
)
axD.grid(axis="x", alpha=0.25, linestyle="--")
axD.grid(axis="y", visible=False)
axD.tick_params(axis="y", labelsize=10)

# -------------------------
# Panel E: summary table
# -------------------------
axE.axis("off")

axE.text(
    0.0,
    1.05,
    "E. Case-study interpretation summary",
    transform=axE.transAxes,
    fontsize=13,
    fontweight="bold",
    ha="left",
    va="bottom"
)

tbl = axE.table(
    cellText=summary10.values,
    colLabels=summary10.columns,
    cellLoc="center",
    colLoc="center",
    bbox=[0.0, 0.0, 1.0, 0.92],
    colWidths=[0.48, 0.52]
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10.5)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("#A6A6A6")
    cell.set_linewidth(0.45)

    if row == 0:
        cell.set_facecolor(blue)
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#F3F6FA" if row % 2 == 1 else "white")
        cell.set_text_props(weight="bold")

fig.suptitle(
    "Figure 10. Case-study interpretation of validated and novel E3/DUB–substrate interactions",
    fontsize=20,
    fontweight="bold",
    y=0.985
)

plt.subplots_adjust(
    top=0.92,
    bottom=0.04,
    left=0.08,
    right=0.96
)

save_final_figure(
    fig,
    10,
    "Figure10_case_study_biological_interpretation"
)

plt.show()

بریم Figure 11 — Graphical Abstract / Final Story Figure.

این شکل برای خلاصه کل پروژه است: از داده خام تا مدل نهایی و کشف کاندیدهای جدید.

In [ ]:
# Figure 11 — Graphical abstract / final project story

fig = plt.figure(figsize=(24, 14), constrained_layout=False)

gs = GridSpec(
    2, 3,
    figure=fig,
    height_ratios=[1.0, 0.85],
    width_ratios=[1, 1, 1],
    hspace=0.35,
    wspace=0.25
)

axA = fig.add_subplot(gs[0, :])
axB = fig.add_subplot(gs[1, 0])
axC = fig.add_subplot(gs[1, 1])
axD = fig.add_subplot(gs[1, 2])

for ax in [axA, axB, axC, axD]:
    ax.axis("off")

blue = "#164A8B"
red = "#D73027"
green = "#7EAD50"
purple = "#7B3294"
orange = "#ED7D31"
gray = "#F3F6FA"
dark = "#222222"

# -------------------------
# Panel A: end-to-end story
# -------------------------
steps = [
    ("Curated interactions", "E3/DUB–substrate\nknown pairs"),
    ("Context-aware negatives", "PPI\nLocalization\nscRNA"),
    ("Protein representations", "ESM2\nProtBERT\nPer-residue"),
    ("Heterogeneous graph", "Interaction\nPPI\nCo-localization"),
    ("HGT ensemble", "MegaWeighted\nfinal model"),
    ("Biological discovery", "Novel candidates\nCancer prioritization"),
]

xs = np.linspace(0.08, 0.92, len(steps))
y = 0.55

step_colors = [blue, blue, orange, purple, red, green]

for i, (title, subtitle) in enumerate(steps):
    x = xs[i]

    box = FancyBboxPatch(
        (x - 0.065, y - 0.17),
        0.13,
        0.34,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        linewidth=1.4,
        edgecolor=dark,
        facecolor=gray,
        transform=axA.transAxes
    )
    axA.add_patch(box)

    axA.add_patch(
        Circle(
            (x, y + 0.21),
            0.035,
            facecolor=step_colors[i],
            edgecolor="black",
            linewidth=0.8,
            transform=axA.transAxes
        )
    )

    axA.text(
        x,
        y + 0.21,
        str(i + 1),
        ha="center",
        va="center",
        color="white",
        fontsize=12,
        fontweight="bold",
        transform=axA.transAxes
    )

    axA.text(
        x,
        y + 0.065,
        title,
        ha="center",
        va="center",
        fontsize=11.5,
        fontweight="bold",
        transform=axA.transAxes
    )

    axA.text(
        x,
        y - 0.065,
        subtitle,
        ha="center",
        va="center",
        fontsize=10,
        linespacing=1.25,
        transform=axA.transAxes
    )

    if i < len(steps) - 1:
        arrow = FancyArrowPatch(
            (x + 0.078, y),
            (xs[i + 1] - 0.078, y),
            arrowstyle="-|>",
            mutation_scale=17,
            linewidth=1.8,
            color=dark,
            transform=axA.transAxes
        )
        axA.add_patch(arrow)

axA.set_title(
    "A. End-to-end E3/DUB interaction discovery framework",
    loc="left",
    fontsize=14,
    fontweight="bold"
)

# -------------------------
# Panel B: final model metrics
# -------------------------
best_model = all_models.sort_values("pr_auc_mean", ascending=False).iloc[0]

metrics = pd.DataFrame({
    "metric": ["ROC-AUC", "PR-AUC", "F1", "Accuracy"],
    "value": [
        best_model["roc_auc_mean"],
        best_model["pr_auc_mean"],
        best_model["f1_mean"],
        best_model["accuracy_mean"],
    ]
})

axB.axis("on")

axB.bar(
    metrics["metric"],
    metrics["value"],
    color=[blue, red, green, orange],
    edgecolor="black",
    linewidth=0.8
)

axB.set_ylim(0, 1)
axB.set_ylabel("Score")
axB.set_title(
    "B. Final ensemble performance",
    loc="left",
    fontweight="bold"
)

for i, r in metrics.iterrows():
    axB.text(
        i,
        r["value"] + 0.025,
        f"{r['value']:.3f}",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

axB.grid(axis="y", alpha=0.25, linestyle="--")
axB.grid(axis="x", visible=False)

# -------------------------
# Panel C: key discoveries
# -------------------------
discoveries = [
    ("Best model", "MegaWeighted HGT ensemble"),
    ("Novel discovery", "Top E3/DUB–substrate candidates"),
    ("Explainability", "PPI-dominant relation dependency"),
    ("Cancer focus", "CRC / LIHC / GI prioritization"),
]

for i, (k, v) in enumerate(discoveries):
    yy = 0.82 - i * 0.22

    box = FancyBboxPatch(
        (0.05, yy - 0.07),
        0.90,
        0.13,
        boxstyle="round,pad=0.015,rounding_size=0.02",
        linewidth=1.0,
        edgecolor=dark,
        facecolor="white",
        transform=axC.transAxes
    )
    axC.add_patch(box)

    axC.text(
        0.10,
        yy + 0.025,
        k,
        ha="left",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=red if i == 0 else dark,
        transform=axC.transAxes
    )

    axC.text(
        0.10,
        yy - 0.035,
        v,
        ha="left",
        va="center",
        fontsize=10,
        transform=axC.transAxes
    )

axC.set_title(
    "C. Main biological and modeling outcomes",
    loc="left",
    fontsize=14,
    fontweight="bold"
)

# -------------------------
# Panel D: final candidate examples
# -------------------------
candidate_examples = [
    "USP15→TNFAIP3",
    "USP10→CTNNB1",
    "USP15→MAPK3",
    "CDC20→TP53",
    "STUB1→FOS",
]

for i, cand in enumerate(candidate_examples):
    yy = 0.82 - i * 0.16

    axD.add_patch(
        FancyBboxPatch(
            (0.08, yy - 0.05),
            0.84,
            0.10,
            boxstyle="round,pad=0.012,rounding_size=0.02",
            linewidth=1.0,
            edgecolor=dark,
            facecolor=gray if i % 2 == 0 else "white",
            transform=axD.transAxes
        )
    )

    axD.text(
        0.50,
        yy,
        cand,
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold",
        color=purple if i < 2 else dark,
        transform=axD.transAxes
    )

axD.set_title(
    "D. Representative high-priority candidates",
    loc="left",
    fontsize=14,
    fontweight="bold"
)

fig.suptitle(
    "Figure 11. Graphical abstract of the E3/DUB specificity prediction framework",
    fontsize=20,
    fontweight="bold",
    y=0.985
)

plt.subplots_adjust(
    top=0.90,
    bottom=0.06,
    left=0.05,
    right=0.98
)

save_final_figure(
    fig,
    11,
    "Figure11_graphical_abstract_E3DUB_framework"
)

plt.show()